# Evaluación Parcial N°3 — Redes recurrentes para clasificación de actos de diálogo

## Clasificación del acto de diálogo en DailyDialog con RNN simple, LSTM y GRU (PyTorch)

**Curso:** DLY0100 — Deep Learning<br>
**Sección:** 003D<br>
**Integrantes:** Esteban Pacheco Vega y Francisco Figueroa Donoso<br>
**Fecha:** noviembre de 2026<br>
**Repositorio:** https://github.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY0100_003D

## 1. Introducción

### 1.1 Descripción del problema

El dataset corresponde a **DailyDialog** (Li et al., 2017): diálogos cotidianos en inglés escritos por personas, divididos en `train` (11.118 diálogos), `validation` (1.000) y `test` (1.000). Cada turno de cada diálogo trae dos etiquetas: el **acto de diálogo** (`act`) y la **emoción** (`emotion`).

El **acto de diálogo** es la función comunicativa del turno, es decir, qué *hace* el hablante con lo que dice:

| Código | Acto | Qué hace el hablante | Ejemplo del dataset |
|---|---|---|---|
| 1 | *inform* | Entrega información o una opinión | «It's Ellie and her friends .» |
| 2 | *question* | Pide información | «Who's Ellie ?» |
| 3 | *directive* | Pide, sugiere u ordena que el otro haga algo | «Say , Jim , how about going for a few beers after dinner ?» |
| 4 | *commissive* | Se compromete, acepta o rechaza | «You know that is tempting but is really not good for our fitness .» |

En este notebook se entrena una red recurrente que recibe el texto de **un turno** y predice su acto de diálogo (clasificación en 4 clases).

**Por qué una red recurrente:** el acto depende del orden de las palabras: «you will call me» informa, mientras «will you call me» pide algo, con exactamente las mismas palabras.

- Un **MLP** recibe un vector de tamaño fijo y trata cada posición como una entrada independiente, sin compartir lo aprendido entre posiciones.
- Una **CNN** detecta patrones locales con sus filtros, pero cada filtro cubre solo unas pocas palabras seguidas.
- Una **RNN** recorre el turno token a token con los mismos pesos, acepta turnos de cualquier largo y acumula el contexto en su estado oculto.

### 1.2 Objetivo del modelo

Entrenar con **PyTorch** una red recurrente que clasifique el acto de cada turno, aplicando:

- Carga, exploración y preprocesamiento del texto (vocabulario propio, relleno y lotes por largo).
- Comparación de **RNN simple, LSTM y GRU** para justificar la elección de la celda.
- **Rango de error entre reentrenamientos**, para distinguir mejoras reales de variaciones aleatorias.
- Experimentos controlados que modifican **un único hiperparámetro a la vez**, con énfasis en **tasa de aprendizaje, tamaño del lote y épocas**, y verificaciones finales de la dependencia del orden.
- Evaluación con exactitud (*accuracy*), precisión (*precision*), exhaustividad (*recall*) y F1-score, global y por clase, frente a dos líneas base sin aprendizaje.

**Nota de terminología:** *accuracy* se traduce como **exactitud** (proporción de turnos bien clasificados), *precision* como **precisión** (de los turnos que el modelo asignó a una clase, cuántos eran de esa clase) y *recall* como **exhaustividad** (de los turnos de una clase, cuántos detectó el modelo).

**Criterio principal: F1 macro.** Las clases están desbalanceadas (*commissive* es solo el 9,3 % de `train`). La exactitud puede ser alta aunque el modelo ignore las clases pequeñas; el F1 macro promedia las cuatro clases con el mismo peso.

### 1.3 Estructura del notebook

1. Introducción.
2. Configuración inicial.
3. Carga y exploración de los datos.
4. Preprocesamiento (vocabulario, codificación y lotes).
5. Utilidades de entrenamiento, evaluación y experimentos.
6. Definición de la arquitectura recurrente.
7. Entrenamiento del modelo base.
8. Rango de error entre reentrenamientos.
9. Experimento clave: tipo de celda (RNN simple vs. LSTM vs. GRU).
10. Experimentos de entrenamiento: tasa de aprendizaje, tamaño del lote y épocas.
11. Experimentos de arquitectura.
12. Experimentos de regularización y desbalance.
13. Modelo final.
14. Evaluación del modelo final en el conjunto de test.
15. Comparación entre la configuración base y la final.
16. Síntesis de los experimentos.
17. Verificaciones de la dependencia del orden.
18. Conclusiones generales y mejoras propuestas.
19. Conclusión final.
20. Referencias.
21. Anexo A: síntesis visual para la presentación.
22. Anexo B: herramientas y tecnologías de apoyo utilizadas.
23. Respaldo de resultados.

**Criterio metodológico:** todas las decisiones se toman con el conjunto de **validación**. El conjunto de **test** se usa una sola vez, al final, para evaluar el modelo elegido. Todas las variantes se entrenan con el mismo tope de épocas y la misma paciencia, incluido el modelo final.

## 2. Configuración inicial

Se importan en un único bloque todas las librerías del notebook:

- **PyTorch** (`torch.nn`, `torch.optim`, `DataLoader`, `Sampler`, `pad_sequence`): da control explícito sobre el ciclo de entrenamiento y aprovecha la GPU.
- **`pack_padded_sequence`**: permite que la red recurrente procese turnos de largo variable sin que el relleno afecte su estado final.
- **NumPy / Pandas**: arreglos y tablas comparativas.
- **scikit-learn**: métricas de evaluación y pesos de clase.
- **Plotly**: gráficos interactivos. El renderizador `vscode+colab` permite verlos tanto en Colab como en VS Code conectado a un servidor de Colab.
- **IPython.display**: tablas con `display`.
- **re / time / random / copy / json / os / shutil / collections**: lectura del texto, tiempos, semillas, copia de los mejores pesos, resultados guardados y respaldo.

Se fija una semilla para que los resultados sean reproducibles en la medida de lo posible.

In [1]:
# ============================================================
# Importación de librerías (todos los imports del notebook van aquí)
# ============================================================
import copy
import json
import os
import random
import re
import shutil
import time
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader
from torch.utils.data.sampler import Sampler

# ============================================================
# Reproducibilidad, dispositivo y visualización
# ============================================================
SEED = 42


def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


fijar_semilla(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pio.renderers.default = "vscode+colab"
pd.set_option("display.max_colwidth", 120)
COLORES = ["#104281", "#3987E5", "#86B6EF"]
AMARILLO = "#F7B500"
print("PyTorch", torch.__version__, "| dispositivo:", device)

PyTorch 2.14.1+cu130 | dispositivo: cuda


**Nota:** aun con semilla fija, el entrenamiento en GPU no es completamente determinista. Por eso, en la sección 8 se mide ese margen de variación antes de comparar configuraciones.

**Requisito de ejecución:** activar GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*).

### 2.1 Reutilización de resultados ya obtenidos

Las corridas de este notebook se ejecutaron previamente con las mismas configuraciones y semillas. Para no repetir cómputo, sus métricas se guardaron en `resultados_rnn_guardados.json` (en el repositorio) y se reutilizan cuando la configuración y la semilla coinciden **exactamente**: la clave de cada corrida incluye todos los hiperparámetros, la semilla, el tope de épocas y la paciencia.

- Las corridas que necesitan el modelo entrenado (modelo base, comparación de celdas por largo, pesos de clase, experimento de épocas y modelo final) se entrenan siempre.
- Cualquier configuración nueva (por ejemplo, una ampliación de rango o una verificación del orden) se entrena.
- Cada fila de resultados indica su `origen`: `entrenado` en esta ejecución o `guardado` de la ejecución anterior. Los valores guardados tienen 4 decimales.
- Con `REUTILIZAR_RESULTADOS = False` se entrena todo desde cero.

In [2]:
# ============================================================
# Carga de los resultados guardados
# ============================================================
REUTILIZAR_RESULTADOS = True
URL_REPO = "https://raw.githubusercontent.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY0100_003D/main/"


def leer_archivo_repo(nombre):
    if os.path.exists(nombre):
        with open(nombre, encoding="utf-8") as f:
            return f.read()
    with urllib.request.urlopen(URL_REPO + nombre) as r:
        return r.read().decode("utf-8")


RESULTADOS_GUARDADOS = json.loads(leer_archivo_repo("resultados_rnn_guardados.json")) if REUTILIZAR_RESULTADOS else {}
CORRIDAS_NUEVAS = {}
print(f"Corridas guardadas disponibles: {len(RESULTADOS_GUARDADOS)}")

Corridas guardadas disponibles: 54


## 3. Carga y exploración de los datos

### 3.1 Lectura de los archivos desde el repositorio

Los tres archivos (`train.csv`, `validation.csv` y `test.csv`) están en la raíz del repositorio del equipo, por lo que se leen directamente desde GitHub. Si existe una copia local, se usa esa.

Cada fila es un **diálogo completo** con tres columnas: `dialog` (lista de turnos), `act` (acto de cada turno, 1 a 4) y `emotion` (emoción de cada turno, 0 a 6).

In [3]:
# ============================================================
# Lectura de las tres particiones
# ============================================================
def cargar_particion(nombre):
    ruta_local = f"{nombre}.csv"
    return pd.read_csv(ruta_local if os.path.exists(ruta_local) else URL_REPO + ruta_local)


df_train = cargar_particion("train")
df_val = cargar_particion("validation")
df_test = cargar_particion("test")

display(pd.DataFrame({
    "particion": ["train", "validation", "test"],
    "dialogos": [len(df_train), len(df_val), len(df_test)],
    "valores_nulos": [int(d.isna().sum().sum()) for d in (df_train, df_val, df_test)],
}))

,particion,dialogos,valores_nulos
0,train,11118,0
1,validation,1000,0
2,test,1000,0


In [4]:
# ============================================================
# Primeras filas de train
# ============================================================
display(df_train.head(3))

,dialog,act,emotion
0,"['Say , Jim , how about going for a few beers after dinner ? '\n ' You know that is tempting but is really not good ...",[3 4 2 2 2 3 4 1 3 4],[0 0 0 0 0 0 4 4 4 4]
1,"['Can you do push-ups ? '\n "" Of course I can . It's a piece of cake ! Believe it or not , I can do 30 push-ups a mi...",[2 1 2 2 1 1],[0 0 6 0 0 0]
2,"['Can you study with the radio on ? '\n ' No , I listen to background music . ' ' What is the difference ? '\n ' The...",[2 1 2 1 1],[0 0 0 0 0]


**Observación:** las tres particiones tienen 11.118, 1.000 y 1.000 diálogos, como indica la documentación del dataset, y ninguna tiene valores nulos.

### 3.2 Lectura de la columna `dialog`

La columna `dialog` **no** es una lista de Python válida: es la representación en texto de un arreglo de NumPy, sin comas entre los turnos, y cada turno va entre comillas simples o dobles según contenga apóstrofos. Por eso `ast.literal_eval` falla.

`parsear_dialogo(texto)` extrae cada turno con una expresión regular que captura el texto entre comillas **respetando las comillas escapadas** (`\'`), que aparecen en algunos turnos (por ejemplo, «She\'s my little niece»). Una versión que no respete los escapes corta esos turnos en dos y deja 5 diálogos de `train` con más turnos que etiquetas.

`parsear_etiquetas(texto)` convierte `[3 4 2 2]` en la lista `[3, 4, 2, 2]`.

In [5]:
# ============================================================
# Parser de turnos y etiquetas
# ============================================================
PATRON_TURNO = re.compile(r"'((?:[^'\\]|\\.)*)'|\"((?:[^\"\\]|\\.)*)\"", re.S)


def parsear_dialogo(texto):
    turnos = [(a or b) for a, b in PATRON_TURNO.findall(texto)]
    return [t.replace("\\'", "'").replace('\\"', '"').strip() for t in turnos]


def parsear_etiquetas(texto):
    return [int(x) for x in texto.strip("[]").split()]


for turno, acto in zip(parsear_dialogo(df_train.loc[0, "dialog"]), parsear_etiquetas(df_train.loc[0, "act"])):
    print(acto, "·", turno)

3 · Say , Jim , how about going for a few beers after dinner ?
4 · You know that is tempting but is really not good for our fitness .
2 · What do you mean ? It will help us to relax .
2 · Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?
2 · I guess you are right.But what shall we do ? I don't feel like sitting at home .
3 · I suggest a walk over to the gym where we can play singsong and meet some of our friends .
4 · That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .
1 · Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .
3 · Good.Let ' s go now .
4 · All right .


### 3.3 Validación y aplanado a nivel de turno

`aplanar(df)` pasa de una fila por diálogo a **una fila por turno**, con las columnas `dialogo_id`, `posicion`, `texto`, `n_tokens`, `acto` y `emocion`. El texto ya viene tokenizado con espacios (la puntuación aparece separada: `after dinner ?`), por lo que basta con pasarlo a minúsculas y separar por espacios.

El acto se reindexa de 1–4 a **0–3**, que es lo que espera `CrossEntropyLoss`. Si un diálogo tuviera distinta cantidad de turnos que de etiquetas, se descarta y se informa.

In [6]:
# ============================================================
# Aplanado de diálogos a turnos
# ============================================================
ACTOS = ["inform", "question", "directive", "commissive"]


def aplanar(df):
    filas, desalineados = [], []
    for i, fila in df.iterrows():
        turnos = parsear_dialogo(fila["dialog"])
        actos = parsear_etiquetas(fila["act"])
        emociones = parsear_etiquetas(fila["emotion"])
        if not (len(turnos) == len(actos) == len(emociones)):
            desalineados.append(i)
            continue
        for pos, (texto, acto, emocion) in enumerate(zip(turnos, actos, emociones)):
            tokens = texto.lower().split()
            filas.append({"dialogo_id": i, "posicion": pos, "texto": " ".join(tokens),
                          "n_tokens": len(tokens), "acto": acto - 1, "emocion": emocion})
    return pd.DataFrame(filas), desalineados


turnos_train, des_train = aplanar(df_train)
turnos_val, des_val = aplanar(df_val)
turnos_test, des_test = aplanar(df_test)

display(pd.DataFrame({
    "particion": ["train", "validation", "test"],
    "turnos": [len(turnos_train), len(turnos_val), len(turnos_test)],
    "dialogos_descartados": [len(des_train), len(des_val), len(des_test)],
    "turnos_vacios": [int((t["n_tokens"] == 0).sum()) for t in (turnos_train, turnos_val, turnos_test)],
}))

,particion,turnos,dialogos_descartados,turnos_vacios
0,train,87170,0,0
1,validation,8069,0,0
2,test,7740,0,0


In [7]:
# ============================================================
# Primeros turnos aplanados
# ============================================================
display(turnos_train.head())

,dialogo_id,posicion,texto,n_tokens,acto,emocion
0,0,0,"say , jim , how about going for a few beers after dinner ?",14,2,0
1,0,1,you know that is tempting but is really not good for our fitness .,14,3,0
2,0,2,what do you mean ? it will help us to relax .,12,1,0
3,0,3,do you really think so ? i don't . it will just make us fat and act silly . remember last time ?,23,1,0
4,0,4,i guess you are right.but what shall we do ? i don't feel like sitting at home .,18,1,0


**Observación:** con el parser que respeta las comillas escapadas, los 13.118 diálogos quedan alineados (0 descartados) y no hay turnos vacíos: 87.170 turnos en `train`, 8.069 en validación y 7.740 en test.

### 3.4 Distribución de clases

Se compara la proporción de cada acto en las tres particiones, en tabla y en un gráfico de barras agrupadas.

In [8]:
# ============================================================
# Distribución del acto de diálogo por partición
# ============================================================
conteos = pd.DataFrame({nombre: t["acto"].value_counts().sort_index()
                        for nombre, t in [("train", turnos_train), ("validation", turnos_val), ("test", turnos_test)]})
conteos.index = ACTOS
tabla_distribucion = conteos.astype(str) + " (" + (conteos / conteos.sum() * 100).round(1).astype(str) + " %)"
display(tabla_distribucion)

,train,validation,test
inform,39873 (45.7 %),3125 (38.7 %),3534 (45.7 %)
question,24974 (28.6 %),2244 (27.8 %),2210 (28.6 %)
directive,14242 (16.3 %),1775 (22.0 %),1278 (16.5 %)
commissive,8081 (9.3 %),925 (11.5 %),718 (9.3 %)


In [9]:
# ============================================================
# Gráfico de la distribución por partición
# ============================================================
porcentajes = ((conteos / conteos.sum() * 100).round(1).reset_index(names="acto")
               .melt(id_vars="acto", var_name="particion", value_name="porcentaje"))
fig = px.bar(porcentajes, x="acto", y="porcentaje", color="particion", barmode="group", text="porcentaje",
             color_discrete_sequence=COLORES, title="Distribución del acto de diálogo por partición",
             labels={"porcentaje": "% de turnos", "particion": "partición"})
fig.show()

**Observación:**

- `train` y `test` tienen la misma distribución: *inform* 45,7 %, *question* 28,6 %, *directive* 16,3–16,5 % y *commissive* 9,3 %. La clase mayor tiene 4,9 veces más turnos que la menor.
- El desbalance justifica usar F1 macro como criterio y probar pesos de clase (sección 12.2).

**Hallazgo:**

- Validación tiene otra distribución: más *directive* (22,0 %) y *commissive* (11,5 %) y menos *inform* (38,7 %). Si esas dos clases resultan más difíciles de clasificar, las métricas de validación quedarán por debajo de las de test. La sección 14.2 confirma que lo son, y así ocurre en todo el notebook.

### 3.5 Largo de los turnos

El largo de los turnos define cuántos tokens debe procesar la red (`maxlen`). Se calcula qué porcentaje de turnos queda completo con distintos valores de `maxlen` y se grafica el histograma.

In [10]:
# ============================================================
# Cobertura de maxlen
# ============================================================
largos = turnos_train["n_tokens"]
print(f"Media: {largos.mean():.1f} | p50: {largos.median():.0f} | p95: {largos.quantile(0.95):.0f} | máximo: {largos.max()}")
display(pd.DataFrame({
    "maxlen": [10, 20, 30, 40],
    "% de turnos completos": [round((largos <= m).mean() * 100, 1) for m in [10, 20, 30, 40]],
}))

Media: 13.6 | p50: 11 | p95: 33 | máximo: 278


,maxlen,% de turnos completos
0,10,48.4
1,20,83.3
2,30,93.7
3,40,97.5


In [11]:
# ============================================================
# Histograma del largo de los turnos
# ============================================================
fig = px.histogram(x=largos.clip(upper=80), nbins=80, color_discrete_sequence=[COLORES[1]],
                   title="Largo de los turnos en train", labels={"x": "tokens por turno (recortado en 80)"})
for valor, etiqueta in [(largos.median(), "p50"), (largos.quantile(0.95), "p95"), (40, "maxlen base")]:
    fig.add_vline(x=valor, line_dash="dash", line_color=COLORES[0], annotation_text=etiqueta)
fig.update_layout(yaxis_title="turnos", showlegend=False)
fig.show()

**Observación:** la mediana es de 11 tokens por turno, el percentil 95 de 33 y el máximo de 278. Con `maxlen = 40` queda completo el 97,5 % de los turnos; el 2,5 % restante se trunca conservando su inicio. El experimento 11.5 muestra que este valor importa: con 20 tokens el F1 macro cae 3,84 pp.

### 3.6 Ejemplos por clase

Tres turnos de cada acto, y una muestra de turnos *directive* con forma de pregunta.

In [12]:
# ============================================================
# Ejemplos de turnos por acto
# ============================================================
ejemplos = turnos_train.groupby("acto").sample(3, random_state=SEED)[["acto", "texto"]]
ejemplos["acto"] = ejemplos["acto"].map(dict(enumerate(ACTOS)))
display(ejemplos.reset_index(drop=True))

,acto,texto
0,inform,"sure . most facials start with a thorough cleaning . then we usually use a toner to invigorate the skin , followed b..."
1,inform,"i really enjoyed tonight's party . thank you , sarah ."
2,inform,it ’ s all religion . take the theory of the big bang.how is it possible that all of the stuff in the universe comes...
3,question,not bad.how about you ? haven't seen you for a long time .
4,question,"please fill out this form , and what's in your package ?"
5,question,which child is this ?
6,directive,"hello , this is mary from the taipei new station , what can i do for you ?"
7,directive,"operator , i want to make a collect overseas phone call to beijing ."
8,directive,"well , i'll accept the price and place an initial order of 10,000 units ."
9,commissive,"thank you , miss kelly ."


In [13]:
# ============================================================
# Turnos directive con forma de pregunta
# ============================================================
directivos_con_pregunta = turnos_train[(turnos_train["acto"] == 2) & turnos_train["texto"].str.contains(r"\?")]
print(f"Turnos directive con '?': {len(directivos_con_pregunta)} de {(turnos_train['acto'] == 2).sum()}")
display(directivos_con_pregunta.sample(5, random_state=SEED)[["texto"]])

Turnos directive con '?': 5384 de 14242


,texto
67934,"hello . this is the operator . can i speak to mr . smith , please ?"
39040,"no , it ’ s pretty good . why not give the landlord a call ?"
40043,really ? did you see the hijacked planes crash into and destroy the twin towers ?
56494,can you tell me how much that'll cost ?
6656,could you show me where it is ?


**Hallazgo:** 5.384 de los 14.242 turnos *directive* de `train` (37,8 %) contienen «?»: piden algo con forma de pregunta («can you tell me how much that'll cost ?», «could you show me where it is ?»). La regla del «?» los clasifica como *question*, así que el modelo debe usar el resto del turno para separarlos. La sección 14.3 muestra que, en la práctica, esta no terminó siendo la confusión principal.

### 3.7 Líneas base sin aprendizaje

Antes de entrenar se calculan dos referencias en validación:

- **Clase mayoritaria:** siempre predice *inform*.
- **Regla del signo `?`:** si el turno contiene `?` predice *question*; si no, *inform*.

Un modelo que no supere la regla del `?` no aprendió nada más que la puntuación. Por eso el preprocesamiento **conserva** la puntuación como token.

`metricas(y_true, y_pred)` calcula las métricas que se usan en todo el notebook: exactitud, precisión, exhaustividad y F1 (promedio macro), más el F1 ponderado por soporte.

In [14]:
# ============================================================
# Métricas y líneas base
# ============================================================
def metricas(y_true, y_pred):
    precision, exhaustividad, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"exactitud": accuracy_score(y_true, y_pred), "precision_macro": precision,
            "exhaustividad_macro": exhaustividad, "f1_macro": f1,
            "f1_ponderado": f1_score(y_true, y_pred, average="weighted", zero_division=0)}


def linea_base_mayoritaria(turnos):
    return np.full(len(turnos), turnos_train["acto"].mode()[0])


def linea_base_pregunta(turnos):
    return np.where(turnos["texto"].str.contains(r"\?"), 1, 0)


tabla_lineas_base_val = pd.DataFrame([
    {"modelo": "Clase mayoritaria", **metricas(turnos_val["acto"], linea_base_mayoritaria(turnos_val))},
    {"modelo": "Regla «?»", **metricas(turnos_val["acto"], linea_base_pregunta(turnos_val))},
]).round(4)
display(tabla_lineas_base_val)

,modelo,exactitud,precision_macro,exhaustividad_macro,f1_macro,f1_ponderado
0,Clase mayoritaria,0.3873,0.0968,0.2500,0.1396,0.2162
1,Regla «?»,0.6519,0.3358,0.4901,0.3974,0.5231


**Observación:**

- Clase mayoritaria: exactitud 0,387 y F1 macro 0,140.
- Regla «?»: exactitud 0,652 y F1 macro 0,397.

La regla alcanza una exactitud razonable usando solo la puntuación, pero su F1 macro es bajo porque nunca predice *directive* ni *commissive*. El piso que el modelo debe superar es un F1 macro de 0,397.

## 4. Preprocesamiento

### 4.1 Vocabulario

`Vocab` asigna un índice a cada token que aparece al menos `min_freq` veces y envía el resto a `<unk>`. Es una clase mínima propia en vez de `torchtext`, librería que fue descontinuada y ya no es compatible con las versiones actuales de PyTorch.

- El vocabulario se construye **solo con `train`**, para no filtrar información de validación ni de test.
- Tokens especiales: `<unk>` (palabra desconocida) y `<pad>` (relleno).
- `min_freq` controla el tamaño del vocabulario: con 1 se conservan ≈ 22.000 tokens, con 2 ≈ 13.400 y con 3 ≈ 9.900. Es uno de los hiperparámetros de la sección 11.

In [15]:
# ============================================================
# Vocabulario
# ============================================================
class Vocab:
    def __init__(self, counter, min_freq=1, specials=("<unk>",)):
        self.itos = list(specials) + sorted(t for t, c in counter.items() if c >= min_freq and t not in specials)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.default_index = self.stoi["<unk>"]

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __len__(self):
        return len(self.itos)

    def get_itos(self):
        return self.itos


ESPECIALES = ("<unk>", "<pad>")
contador_train = Counter(tok for texto in turnos_train["texto"] for tok in texto.split())
print(f"Tokens distintos en train: {len(contador_train):,}")
for min_freq in [1, 2, 3]:
    print(f"  min_freq = {min_freq}: vocabulario de {len(Vocab(contador_train, min_freq, ESPECIALES)):,} tokens")

Tokens distintos en train: 22,075
  min_freq = 1: vocabulario de 22,077 tokens
  min_freq = 2: vocabulario de 13,373 tokens
  min_freq = 3: vocabulario de 9,848 tokens


### 4.2 Codificación de los turnos

`preparar_datos(maxlen, min_freq)` construye el vocabulario y convierte cada turno en una tupla `(índices, largo, acto)`. Los turnos se truncan a `maxlen` tokens conservando el **inicio**, donde suele estar la señal del acto. El resultado se guarda en caché para no recalcularlo en cada experimento.

In [16]:
# ============================================================
# Codificación de los turnos (con caché por maxlen y min_freq)
# ============================================================
_cache_datos = {}


def preparar_datos(maxlen, min_freq):
    clave = (maxlen, min_freq)
    if clave not in _cache_datos:
        vocab = Vocab(contador_train, min_freq=min_freq, specials=ESPECIALES)

        def codificar(turnos):
            datos = []
            for texto, acto in zip(turnos["texto"], turnos["acto"]):
                ids = [vocab[t] for t in texto.split()[:maxlen]] or [vocab["<unk>"]]
                datos.append((torch.tensor(ids), len(ids), int(acto)))
            return datos

        _cache_datos[clave] = (vocab, codificar(turnos_train), codificar(turnos_val), codificar(turnos_test))
    return _cache_datos[clave]


vocab_ej, datos_train_ej, _, _ = preparar_datos(40, 3)
indices, largo, acto = datos_train_ej[0]
print("Texto        :", turnos_train.loc[0, "texto"])
print("Índices      :", indices.tolist())
print("Reconstruido :", " ".join(vocab_ej.get_itos()[i] for i in indices.tolist()))
print("Acto         :", ACTOS[acto])

unk = vocab_ej["<unk>"]
tokens_val = [t for texto in turnos_val["texto"] for t in texto.split()]
print(f"Tokens de validación fuera del vocabulario (min_freq = 3): {np.mean([vocab_ej[t] == unk for t in tokens_val]):.2%}")

Texto        : say , jim , how about going for a few beers after dinner ?
Índices      : [7553, 13, 4756, 13, 4323, 293, 3857, 3585, 281, 3421, 1008, 447, 2625, 279]
Reconstruido : say , jim , how about going for a few beers after dinner ?
Acto         : directive
Tokens de validación fuera del vocabulario (min_freq = 3): 1.80%


### 4.3 Lotes por largo y relleno

`BucketSampler` arma lotes con turnos de **largo parecido**, para que el relleno sea mínimo y cada lote se procese más rápido. En entrenamiento, el orden de los lotes se baraja en cada época.

`crear_collate(pad_idx)` rellena cada lote con `<pad>` (`pad_sequence`) y entrega también el largo real de cada turno, que la red usa para ignorar el relleno.

Para validación y test se usa un `DataLoader` secuencial (`crear_loader_eval`), que conserva el orden original y permite cruzar cada predicción con su texto.

In [17]:
# ============================================================
# BucketSampler y DataLoaders
# ============================================================
class BucketSampler(Sampler):
    def __init__(self, batch_size, datos, semilla=SEED):
        self.rng = random.Random(semilla)
        indices = [(i, d[1]) for i, d in enumerate(datos)]
        self.rng.shuffle(indices)
        agrupados = []
        for i in range(0, len(indices), batch_size * 100):
            agrupados.extend(sorted(indices[i:i + batch_size * 100], key=lambda x: x[1], reverse=True))
        self.lotes = [[idx for idx, _ in agrupados[i:i + batch_size]] for i in range(0, len(agrupados), batch_size)]

    def __iter__(self):
        self.rng.shuffle(self.lotes)
        return iter(self.lotes)

    def __len__(self):
        return len(self.lotes)


def crear_collate(pad_idx):
    def collate_batch(lote):
        secuencias = [ids for ids, _, _ in lote]
        largos = torch.tensor([largo for _, largo, _ in lote])
        etiquetas = torch.tensor([y for _, _, y in lote])
        return pad_sequence(secuencias, batch_first=True, padding_value=pad_idx), largos, etiquetas
    return collate_batch


def crear_loader_train(datos, batch_size, pad_idx, semilla=SEED):
    return DataLoader(datos, batch_sampler=BucketSampler(batch_size, datos, semilla), collate_fn=crear_collate(pad_idx))


def crear_loader_eval(datos, pad_idx, batch_size=512):
    return DataLoader(datos, batch_size=batch_size, shuffle=False, collate_fn=crear_collate(pad_idx))


x, largos_lote, y = next(iter(crear_loader_train(datos_train_ej, 64, vocab_ej["<pad>"])))
print("Lote:", tuple(x.shape), "| largos:", largos_lote[:8].tolist(), "| etiquetas:", y[:8].tolist())

Lote: (64, 11) | largos: [11, 11, 11, 11, 11, 11, 11, 11] | etiquetas: [1, 2, 1, 0, 3, 0, 0, 0]


### 4.4 Pesos de clase

Para el experimento de desbalance (12.2) se calculan pesos **balanceados**: cada clase pesa en la pérdida en proporción inversa a su frecuencia en `train`.

In [18]:
# ============================================================
# Pesos de clase balanceados
# ============================================================
PESOS_CLASE = compute_class_weight("balanced", classes=np.arange(4), y=turnos_train["acto"])
display(pd.DataFrame({"acto": ACTOS, "peso": PESOS_CLASE.round(3)}))

,acto,peso
0,inform,0.547
1,question,0.873
2,directive,1.530
3,commissive,2.697


## 5. Utilidades de entrenamiento, evaluación y experimentos

### 5.1 Una época de entrenamiento y predicción

- `entrenar_epoca(...)`: recorre los lotes, calcula la pérdida, hace *backpropagation*, recorta el gradiente (`clip_grad_norm_`, para evitar saltos bruscos en las actualizaciones) y actualiza los pesos. Devuelve la pérdida y la exactitud de entrenamiento.
- `predecir(modelo, loader, criterio)`: evalúa sin gradientes y devuelve la pérdida, las etiquetas reales, las predichas, las probabilidades y el largo de cada turno.

In [19]:
# ============================================================
# Entrenamiento de una época y predicción
# ============================================================
def entrenar_epoca(modelo, loader, optimizador, criterio, clip):
    modelo.train()
    perdida, aciertos, total = 0.0, 0, 0
    for x, largos, y in loader:
        x, y = x.to(device), y.to(device)
        optimizador.zero_grad()
        logits = modelo(x, largos)
        loss = criterio(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), clip)
        optimizador.step()
        perdida += loss.item() * len(y)
        aciertos += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return perdida / total, aciertos / total


def predecir(modelo, loader, criterio=None):
    modelo.eval()
    perdida, total = 0.0, 0
    y_true, y_pred, probas, largos_todos = [], [], [], []
    with torch.no_grad():
        for x, largos, y in loader:
            logits = modelo(x.to(device), largos)
            if criterio is not None:
                perdida += criterio(logits, y.to(device)).item() * len(y)
            total += len(y)
            y_true += y.tolist()
            y_pred += logits.argmax(1).cpu().tolist()
            probas.append(torch.softmax(logits, 1).cpu().numpy())
            largos_todos += largos.tolist()
    return {"perdida": perdida / total, "y_true": np.array(y_true), "y_pred": np.array(y_pred),
            "probas": np.concatenate(probas), "largos": np.array(largos_todos)}

### 5.2 Entrenamiento con Early Stopping y registro de métricas

`entrenar_y_registrar(config, semilla, epocas, paciencia, verbose, forzar)` construye el modelo a partir de un diccionario de configuración y lo entrena con **Early Stopping** manual: monitorea la pérdida de validación y, al terminar, restaura los pesos de la mejor época. Con `paciencia=0` entrena todas las épocas sin detenerse.

- **Régimen uniforme:** todas las corridas usan `EPOCAS_EXPERIMENTO = 25` y `PACIENCIA_EXPERIMENTO = 3`, incluido el modelo final. Si una variante llegara al tope todavía mejorando, se informa para extenderla.
- Si existe un resultado guardado con la misma clave (sección 2.1) y `forzar=False`, se reutiliza y no se entrena; en ese caso no hay modelo ni historia.
- La pérdida de validación se calcula **sin pesos de clase**, para que sea comparable entre variantes.
- **Retorna** `(fila, historia, modelo)`. `fila` contiene las métricas de la **mejor época**: exactitud de entrenamiento y de validación, F1 macro de validación, pérdida de validación, brecha entre entrenamiento y validación (indicador de sobreajuste), mejor época, épocas entrenadas, parámetros, tiempos y origen.

In [20]:
# ============================================================
# Entrenamiento con Early Stopping y registro de métricas
# ============================================================
EPOCAS_EXPERIMENTO = 25
PACIENCIA_EXPERIMENTO = 3
CRITERIO_EVAL = nn.CrossEntropyLoss()


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


def clave_corrida(config, semilla, epocas, paciencia):
    return json.dumps({**config, "semilla": semilla, "epocas": epocas, "paciencia": paciencia}, sort_keys=True)


def entrenar_y_registrar(config, semilla=SEED, epocas=None, paciencia=None, verbose=False, forzar=False):
    epocas = epocas or EPOCAS_EXPERIMENTO
    paciencia = PACIENCIA_EXPERIMENTO if paciencia is None else paciencia
    clave = clave_corrida(config, semilla, epocas, paciencia)
    if not forzar and clave in RESULTADOS_GUARDADOS:
        return {**RESULTADOS_GUARDADOS[clave], "origen": "guardado"}, None, None

    vocab, datos_train, datos_val, _ = preparar_datos(config["maxlen"], config["min_freq"])
    pad_idx = vocab["<pad>"]
    fijar_semilla(semilla)
    loader_train = crear_loader_train(datos_train, config["batch_size"], pad_idx, semilla)
    loader_val = crear_loader_eval(datos_val, pad_idx)
    modelo = construir_modelo(config, len(vocab), pad_idx)
    pesos = torch.tensor(PESOS_CLASE, dtype=torch.float, device=device) if config["pesos_clase"] else None
    criterio = nn.CrossEntropyLoss(weight=pesos)
    optimizador = optim.Adam(modelo.parameters(), lr=config["learning_rate"])

    historia = {"loss_train": [], "exactitud_train": [], "loss_val": [], "exactitud_val": [], "f1_val": []}
    mejor_loss, mejor_estado, mejor_epoca, sin_mejora = float("inf"), None, 0, 0
    inicio = time.time()
    for epoca in range(epocas):
        loss_tr, acc_tr = entrenar_epoca(modelo, loader_train, optimizador, criterio, config["clip"])
        val = predecir(modelo, loader_val, CRITERIO_EVAL)
        acc_val = accuracy_score(val["y_true"], val["y_pred"])
        f1_val = f1_score(val["y_true"], val["y_pred"], average="macro")
        for k, valor in zip(historia, [loss_tr, acc_tr, val["perdida"], acc_val, f1_val]):
            historia[k].append(valor)
        if verbose:
            print(f"Época {epoca + 1:02} | loss train {loss_tr:.4f} | loss val {val['perdida']:.4f} "
                  f"| exactitud val {acc_val:.4f} | F1 macro val {f1_val:.4f}")
        if val["perdida"] < mejor_loss:
            mejor_loss, mejor_epoca, sin_mejora = val["perdida"], epoca, 0
            mejor_estado = copy.deepcopy(modelo.state_dict())
        else:
            sin_mejora += 1
            if paciencia and sin_mejora >= paciencia:
                break
    duracion = time.time() - inicio
    modelo.load_state_dict(mejor_estado)
    if paciencia and mejor_epoca >= epocas - 2:
        print(f"  Aviso: la mejor época ({mejor_epoca + 1}) está al final del tope; conviene extender las épocas.")

    m = mejor_epoca
    fila = {
        "exactitud_train": historia["exactitud_train"][m],
        "exactitud_val": historia["exactitud_val"][m],
        "f1_macro_val": historia["f1_val"][m],
        "loss_val": historia["loss_val"][m],
        "brecha_train_val": historia["exactitud_train"][m] - historia["exactitud_val"][m],
        "mejor_epoca": m + 1,
        "epocas_entrenadas": len(historia["loss_train"]),
        "parametros": contar_parametros(modelo),
        "segundos": duracion,
        "segundos_por_epoca": duracion / len(historia["loss_train"]),
        "origen": "entrenado",
    }
    CORRIDAS_NUEVAS[clave] = {k: v for k, v in fila.items() if k != "origen"}
    return fila, historia, modelo

### 5.3 Experimentos controlados, selección y ampliación del rango

- `ejecutar_experimento(nombre, variantes, config_actual, forzar)` entrena (o recupera) las **variantes** `{etiqueta: cambios}` que modifican un solo hiperparámetro respecto de la configuración actual, y devuelve una tabla ordenada por **F1 macro de validación**, con la diferencia respecto de la mejor (`diferencia_vs_mejor`) y si cae dentro del rango de error (`dentro_del_margen`, sección 8). Los modelos entrenados quedan en `modelos_ultimo_experimento`.
- `ampliar_rango(...)`: si la mejor variante queda en el **borde** del rango probado, agrega el valor siguiente en esa dirección y lo entrena; repite mientras el ganador siga en el borde y no se supere el límite indicado. Así ningún óptimo queda fuera del rango sin justificar.
- `seleccionar_variante(tabla)` propone la variante con mejor F1 y lista las que están en **empate técnico**. Los empates se deciden caso a caso según el contexto; la decisión queda en la línea `# Cambio Manual` y se justifica en la celda de resultado.
- `aplicar_variante(...)` incorpora la elección a `CONFIG_ACTUAL` y la registra en `decisiones`.
- Si se vuelve a ejecutar un experimento (por ejemplo, para cambiar una decisión), `ejecutar_experimento` restaura la configuración que existía antes de ese experimento y descarta las decisiones y tablas posteriores, que deben volver a ejecutarse desde ahí hacia abajo. Así la cadena de decisiones no mezcla valores de experimentos posteriores.

In [21]:
# ============================================================
# Experimentos controlados (un hiperparámetro a la vez)
# ============================================================
resultados_experimentos = []
historial_experimentos = []
decisiones = {}
modelos_ultimo_experimento = {}
COLUMNAS_TABLA = ["variante", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val",
                  "mejor_epoca", "parametros", "segundos_por_epoca", "diferencia_vs_mejor", "dentro_del_margen", "origen"]


def entrenar_variantes(nombre, variantes, config_actual, forzar=False):
    filas = []
    for etiqueta, cambios in variantes.items():
        config = {**config_actual, **cambios}
        fila, _, modelo = entrenar_y_registrar(config, forzar=forzar)
        print(f"[{nombre}] {etiqueta}: {fila['origen']}")
        if modelo is not None:
            modelos_ultimo_experimento[etiqueta] = (config, modelo)
        filas.append({"experimento": nombre, "variante": etiqueta, **fila})
    return filas


def armar_tabla(filas):
    tabla = pd.DataFrame(filas)
    tabla["diferencia_vs_mejor"] = tabla["f1_macro_val"].max() - tabla["f1_macro_val"]
    tabla["dentro_del_margen"] = tabla["diferencia_vs_mejor"] <= MARGEN_ERROR
    return tabla.sort_values("f1_macro_val", ascending=False).reset_index(drop=True)


def ejecutar_experimento(nombre, variantes, config_actual, forzar=False):
    nombres = [n for n, _ in historial_experimentos]
    if nombre in nombres:
        i = nombres.index(nombre)
        config_actual.clear()
        config_actual.update(historial_experimentos[i][1])
        for n in nombres[i:]:
            decisiones.pop(n, None)
        del historial_experimentos[i:]
        del resultados_experimentos[i:]
        print(f"[{nombre}] Experimento repetido: la configuración vuelve a su estado previo "
              f"y se descartan las decisiones posteriores ({', '.join(nombres[i:])}).")
    historial_experimentos.append((nombre, dict(config_actual)))
    modelos_ultimo_experimento.clear()
    tabla = armar_tabla(entrenar_variantes(nombre, variantes, config_actual, forzar))
    resultados_experimentos.append(tabla)
    return tabla


def ampliar_rango(nombre, variantes, config_actual, hiperparametro, arriba, abajo, minimo, maximo):
    tabla = resultados_experimentos[-1]
    while True:
        valores = {etq: cambios[hiperparametro] for etq, cambios in variantes.items()}
        mejor = valores[tabla.loc[0, "variante"]]
        if mejor == max(valores.values()) and arriba(mejor) <= maximo:
            nuevo = arriba(mejor)
        elif mejor == min(valores.values()) and abajo(mejor) >= minimo:
            nuevo = abajo(mejor)
        else:
            return tabla
        print(f"[{nombre}] La mejor variante está en el borde del rango: se agrega {hiperparametro} = {nuevo}")
        variantes[str(nuevo)] = {hiperparametro: nuevo}
        filas = tabla.drop(columns=["diferencia_vs_mejor", "dentro_del_margen"]).to_dict("records")
        filas += entrenar_variantes(nombre, {str(nuevo): {hiperparametro: nuevo}}, config_actual)
        tabla = armar_tabla(filas)
        resultados_experimentos[-1] = tabla


def seleccionar_variante(tabla):
    mejor = tabla.loc[0, "variante"]
    empatadas = tabla.loc[tabla["dentro_del_margen"] & (tabla["variante"] != mejor), "variante"].tolist()
    print(f"Mejor F1 macro de validación: {mejor}")
    if empatadas:
        print(f"En empate técnico (dentro de {MARGEN_ERROR * 100:.2f} pp): {', '.join(map(str, empatadas))}")
    return mejor


def aplicar_variante(nombre, variantes, elegida):
    CONFIG_ACTUAL.update(variantes[elegida])
    decisiones[nombre] = elegida
    mostrar_config(CONFIG_ACTUAL)

### 5.4 Curvas de entrenamiento y configuración

- `graficar_curvas(historia, titulo)`: pérdida, exactitud y F1 macro por época en tres paneles horizontales, con la mejor época marcada.
- `mostrar_config(config)`: muestra la configuración como tabla vertical.

In [22]:
# ============================================================
# Visualización de curvas y configuración
# ============================================================
def graficar_curvas(historia, titulo):
    epocas = list(range(1, len(historia["loss_train"]) + 1))
    mejor = int(np.argmin(historia["loss_val"])) + 1
    fig = make_subplots(rows=1, cols=3, subplot_titles=["Pérdida", "Exactitud", "F1 macro (validación)"])
    for col, tr, va in [(1, "loss_train", "loss_val"), (2, "exactitud_train", "exactitud_val")]:
        fig.add_trace(go.Scatter(x=epocas, y=historia[tr], name="entrenamiento", line_color=COLORES[2],
                                 legendgroup="train", showlegend=col == 1), row=1, col=col)
        fig.add_trace(go.Scatter(x=epocas, y=historia[va], name="validación", line_color=COLORES[0],
                                 legendgroup="val", showlegend=col == 1), row=1, col=col)
    fig.add_trace(go.Scatter(x=epocas, y=historia["f1_val"], name="F1 macro validación", line_color=COLORES[1]), row=1, col=3)
    for col in [1, 2, 3]:
        fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
        fig.update_xaxes(title_text="época", row=1, col=col)
    fig.update_layout(title=f"{titulo} · mejor época: {mejor}", height=420)
    fig.show()


def mostrar_config(config):
    display(pd.DataFrame({"valor": pd.Series(config).astype(str)}))

## 6. Definición de la arquitectura recurrente

### 6.1 Clasificador recurrente

`Embedding` → capa(s) recurrente(s) → `Dropout` → `Linear(4)`

- **`nn.Embedding`**: convierte cada índice en un vector denso que se aprende durante el entrenamiento. `padding_idx` hace que el vector de `<pad>` quede fijo en cero.
- **Capa recurrente** (`nn.RNN`, `nn.LSTM` o `nn.GRU`): recorre el turno token a token y actualiza su estado oculto hₜ.
  - **RNN simple:** hₜ = tanh(W·xₜ + U·hₜ₋₁ + b). Sufre desvanecimiento del gradiente en secuencias largas.
  - **LSTM:** agrega un estado de celda y compuertas de olvido, entrada y salida que deciden qué conservar.
  - **GRU:** simplifica la LSTM a dos compuertas (actualización y reinicio), con menos parámetros.
- **`pack_padded_sequence`**: le indica a la red el largo real de cada turno, para que el estado final corresponda al último token real y no al relleno.
- Con `bidireccional=True` el turno se lee en ambos sentidos y se concatenan los dos estados finales. Es válido aquí porque se clasifica el turno completo (no se genera texto).
- La salida son 4 **logits**; `CrossEntropyLoss` aplica internamente el softmax.

In [23]:
# ============================================================
# Clasificador recurrente
# ============================================================
CELDAS = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}


class ClasificadorRecurrente(nn.Module):
    def __init__(self, vocab_size, embed_dim, units, capas, celda, bidireccional, dropout, pad_idx, n_clases=4):
        super().__init__()
        self.bidireccional = bidireccional
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = CELDAS[celda](embed_dim, units, num_layers=capas, batch_first=True,
                                 bidirectional=bidireccional, dropout=dropout if capas > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(units * (2 if bidireccional else 1), n_clases)

    def forward(self, x, largos):
        # x = [batch size, seq len]
        emb = self.embedding(x)
        # emb = [batch size, seq len, embed dim]
        empaquetado = pack_padded_sequence(emb, largos.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(empaquetado)
        if isinstance(h, tuple):
            h = h[0]  # la LSTM devuelve (h, c)
        # h = [capas * direcciones, batch size, units]
        final = torch.cat([h[-2], h[-1]], dim=1) if self.bidireccional else h[-1]
        return self.fc(self.dropout(final))


def construir_modelo(config, vocab_size, pad_idx):
    return ClasificadorRecurrente(vocab_size, config["embed_dim"], config["units"], config["capas"],
                                  config["celda"], config["bidireccional"], config["dropout"], pad_idx).to(device)

### 6.2 Configuración base

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `celda` | RNN simple | Punto de partida más simple; la sección 9 la compara con LSTM y GRU |
| `units` | 64 | Estado oculto moderado para turnos de 11 tokens en la mediana |
| `capas` | 1 | Se prueba apilar en la sección 11 |
| `bidireccional` | False | Se prueba en la sección 11 |
| `embed_dim` | 128 | Valor intermedio para ≈ 9.900 tokens |
| `dropout` | 0,0 | Sin regularización inicial; se prueba en la sección 12 |
| `learning_rate` | 0,001 | Valor por defecto de Adam (Kingma y Ba, 2015) |
| `batch_size` | 64 | Equilibrio entre ruido del gradiente y actualizaciones por época (1.362 por época) |
| `maxlen` | 40 | Cubre ≈ 97,5 % de los turnos completos (sección 3.5) |
| `min_freq` | 3 | Vocabulario de ≈ 9.900 tokens (sección 4.1) |
| `pesos_clase` | False | Se prueba en la sección 12 |
| `clip` | 1 | Limita la norma del gradiente para evitar actualizaciones inestables en redes recurrentes |

In [24]:
# ============================================================
# Configuración base
# ============================================================
CONFIG_BASE = dict(
    celda="RNN", units=64, capas=1, bidireccional=False, embed_dim=128, dropout=0.0,
    learning_rate=0.001, batch_size=64, maxlen=40, min_freq=3, pesos_clase=False, clip=1.0,
)
mostrar_config(CONFIG_BASE)

,valor
celda,RNN
units,64
capas,1
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


## 7. Entrenamiento del modelo base

Se entrena `CONFIG_BASE` con la semilla principal (siempre se entrena, para obtener sus curvas).

In [25]:
# ============================================================
# Entrenamiento del modelo base
# ============================================================
fila_base, historia_base, modelo_base = entrenar_y_registrar(CONFIG_BASE, verbose=True, forzar=True)
print(modelo_base)
print(f"Parámetros entrenables: {fila_base['parametros']:,}")

Época 01 | loss train 0.7000 | loss val 0.7120 | exactitud val 0.7288 | F1 macro val 0.6297
Época 02 | loss train 0.5922 | loss val 0.6762 | exactitud val 0.7404 | F1 macro val 0.6398
Época 03 | loss train 0.5491 | loss val 0.6541 | exactitud val 0.7483 | F1 macro val 0.6436
Época 04 | loss train 0.5159 | loss val 0.6527 | exactitud val 0.7601 | F1 macro val 0.6670
Época 05 | loss train 0.4901 | loss val 0.6247 | exactitud val 0.7603 | F1 macro val 0.6975
Época 06 | loss train 0.4648 | loss val 0.6387 | exactitud val 0.7664 | F1 macro val 0.6876
Época 07 | loss train 0.4435 | loss val 0.6406 | exactitud val 0.7661 | F1 macro val 0.6985
Época 08 | loss train 0.4200 | loss val 0.6462 | exactitud val 0.7668 | F1 macro val 0.6986
ClasificadorRecurrente(
  (embedding): Embedding(9848, 128, padding_idx=1)
  (rnn): RNN(128, 64, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=64, out_features=4, bias=True)
)
Parámetros entrenables: 1,273,220


In [26]:
# ============================================================
# Métricas del modelo base
# ============================================================
display(pd.DataFrame([fila_base]).round(4))

,exactitud_train,exactitud_val,f1_macro_val,loss_val,brecha_train_val,mejor_epoca,epocas_entrenadas,parametros,segundos,segundos_por_epoca,origen
0,0.8192,0.7603,0.6975,0.6247,0.0589,5,8,1273220,49.3392,6.1674,entrenado


In [27]:
# ============================================================
# Curvas del modelo base
# ============================================================
graficar_curvas(historia_base, "Modelo base (RNN simple)")

**Observación:**

- La RNN simple obtiene en validación una exactitud de 0,760 y un F1 macro de 0,698: 10,8 pp de exactitud y 30,0 pp de F1 macro sobre la regla «?». Ya aprende bastante más que la puntuación.
- Su mejor época es la 5 de 8 (el Early Stopping se detuvo en la 8), con una brecha de 5,9 pp y unos 6 s por época.
- Desde la época 6 la pérdida de validación sube mientras la de entrenamiento sigue bajando: es el inicio del sobreajuste.

## 8. Rango de error entre reentrenamientos

Entrenar dos veces la misma configuración no entrega el mismo resultado: cambian los pesos iniciales y el orden de los lotes. Si no se conoce el tamaño de esa variación, una diferencia de 0,5 puntos entre dos configuraciones podría interpretarse como mejora cuando es ruido.

Se reentrena `CONFIG_BASE` con otras cuatro semillas (5 en total) y se calculan la media, la desviación estándar, el mínimo, el máximo y el **rango** (máximo − mínimo) del F1 macro de validación. Ese rango se guarda en `MARGEN_ERROR` y se usa en todo el notebook: dos variantes cuya diferencia sea menor o igual se consideran en **empate técnico**.

In [28]:
# ============================================================
# Reentrenamientos de la configuración base con distintas semillas
# ============================================================
SEMILLAS_ADICIONALES = [1, 2, 3, 4]

filas_variabilidad = [{"semilla": SEED, **fila_base}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_BASE, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_variabilidad.append({"semilla": semilla, **fila})
tabla_variabilidad = pd.DataFrame(filas_variabilidad)

Semilla 1: guardado
Semilla 2: guardado
Semilla 3: guardado
Semilla 4: guardado


In [29]:
# ============================================================
# Resultados por semilla
# ============================================================
display(tabla_variabilidad.round(4))

,semilla,exactitud_train,exactitud_val,f1_macro_val,loss_val,brecha_train_val,mejor_epoca,epocas_entrenadas,parametros,segundos,segundos_por_epoca,origen
0,42,0.8192,0.7603,0.6975,0.6247,0.0589,5,8,1273220,49.3392,6.1674,entrenado
1,1,0.8207,0.7635,0.6872,0.6276,0.0571,5,8,1273220,50.3229,6.2904,guardado
2,2,0.7979,0.7529,0.6740,0.6274,0.0450,3,6,1273220,37.1704,6.1951,guardado
3,3,0.8301,0.7661,0.6809,0.6343,0.0639,6,9,1273220,56.2874,6.2542,guardado
4,4,0.8200,0.7629,0.6884,0.6340,0.0571,5,8,1273220,48.9144,6.1143,guardado


In [30]:
# ============================================================
# Estadísticos de variación y rango de error
# ============================================================
columnas_variacion = ["f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca"]
estadisticos_variabilidad = tabla_variabilidad[columnas_variacion].agg(["mean", "std", "min", "max"]).T
estadisticos_variabilidad["rango"] = estadisticos_variabilidad["max"] - estadisticos_variabilidad["min"]
MARGEN_ERROR = estadisticos_variabilidad.loc["f1_macro_val", "rango"]
media_f1_base = estadisticos_variabilidad.loc["f1_macro_val", "mean"]
print(f"F1 macro de validación del modelo base: {media_f1_base:.4f} ± {MARGEN_ERROR / 2:.4f}")
print(f"Rango de error entre reentrenamientos (MARGEN_ERROR): {MARGEN_ERROR:.4f} ({MARGEN_ERROR * 100:.2f} pp)")
display(estadisticos_variabilidad.round(4))

F1 macro de validación del modelo base: 0.6856 ± 0.0118
Rango de error entre reentrenamientos (MARGEN_ERROR): 0.0235 (2.35 pp)


,mean,std,min,max,rango
f1_macro_val,0.6856,0.0088,0.6740,0.6975,0.0235
exactitud_val,0.7611,0.0050,0.7529,0.7661,0.0132
loss_val,0.6296,0.0043,0.6247,0.6343,0.0096
brecha_train_val,0.0564,0.0070,0.0450,0.0639,0.0189
mejor_epoca,4.8000,1.0954,3.0000,6.0000,3.0000


**Observación:**

- Con 5 semillas, el F1 macro del modelo base varía entre 0,674 y 0,698: media 0,686, desviación estándar 0,88 pp y rango de **2,35 pp** (`MARGEN_ERROR`). Desde aquí, toda diferencia de 2,35 pp o menos se considera empate técnico.
- La mejor época varía entre 3 y 6. La pérdida de validación es más estable que el F1 (rango de 0,0096).

## 9. Experimento clave: tipo de celda (RNN simple vs. LSTM vs. GRU)

Desde aquí cada experimento cambia **un único hiperparámetro** respecto de `CONFIG_ACTUAL`. Al terminar, la variante elegida se incorpora a `CONFIG_ACTUAL`. Cada experimento se divide en tres celdas: entrenamiento, tabla y selección. Como la elección secuencial depende del orden, la sección 17 verifica las combinaciones sospechosas al final.

In [31]:
# ============================================================
# Configuración actual (se actualiza al final de cada experimento)
# ============================================================
CONFIG_ACTUAL = dict(CONFIG_BASE)
mostrar_config(CONFIG_ACTUAL)

,valor
celda,RNN
units,64
capas,1
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


### 9.1 Comparación de celdas

Se comparan las tres celdas recurrentes con el resto de la configuración fijo. Además del F1 y la exactitud, se registran los **parámetros** y el **tiempo por época**. Se entrenan siempre, porque sus modelos se usan en la sección 9.2.

In [32]:
# ============================================================
# Experimento 1: celda
# ============================================================
variantes_celda = {
    "RNN simple": dict(celda="RNN"),
    "LSTM": dict(celda="LSTM"),
    "GRU": dict(celda="GRU"),
}
tabla_celda = ejecutar_experimento("celda", variantes_celda, CONFIG_ACTUAL, forzar=True)

[celda] RNN simple: entrenado
[celda] LSTM: entrenado
[celda] GRU: entrenado


In [33]:
# ============================================================
# Resultados del experimento 1
# ============================================================
display(tabla_celda[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,LSTM,0.7154,0.7764,0.5886,0.0766,5,1310468,6.3062,0.0000,True,entrenado
1,GRU,0.7082,0.7793,0.5818,0.0582,4,1298052,6.3108,0.0072,True,entrenado
2,RNN simple,0.6975,0.7603,0.6247,0.0589,5,1273220,5.9835,0.0179,True,entrenado


In [34]:
# ============================================================
# Selección de la variante del experimento 1
# ============================================================
elegida = seleccionar_variante(tabla_celda)
# elegida = "LSTM"  # Cambio Manual
aplicar_variante("celda", variantes_celda, elegida)

Mejor F1 macro de validación: LSTM
En empate técnico (dentro de 2.35 pp): GRU, RNN simple


,valor
celda,LSTM
units,64
capas,1
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:**

- LSTM obtiene el mayor F1 macro (0,7154), seguida de GRU (0,7082) y RNN simple (0,6975). La diferencia entre la mejor y la peor (1,79 pp) está dentro del rango de error: en el total de turnos, las tres celdas empatan.
- Las tres tienen una cantidad de parámetros parecida (1,27–1,31 M), porque casi todos están en el embedding (9.848 × 128 = 1,26 M), y un tiempo por época similar (5,8–6,2 s).
- Se elige la LSTM por mayor F1 y por su ventaja en turnos largos (sección 9.2). La elección se revisa en la sección 17.1.

### 9.2 Desempeño según el largo del turno

La ventaja teórica de las compuertas aparece en **dependencias largas**: en un turno largo, la señal del acto («would you…», «can i…») queda lejos del final, y la RNN simple tiende a olvidarla por el desvanecimiento del gradiente. Se calcula el F1 macro de validación de cada celda por tramo de largo (≤ 10, 11–20 y más de 20 tokens).

In [35]:
# ============================================================
# F1 macro por tramo de largo para cada celda
# ============================================================
TRAMOS = ["≤ 10", "11–20", "> 20"]
tramos = pd.cut(turnos_val["n_tokens"], bins=[0, 10, 20, np.inf], labels=TRAMOS).values
filas_tramos = []
for etiqueta, (config, modelo) in modelos_ultimo_experimento.items():
    vocab, _, datos_val, _ = preparar_datos(config["maxlen"], config["min_freq"])
    pred = predecir(modelo, crear_loader_eval(datos_val, vocab["<pad>"]))
    for tramo in TRAMOS:
        m = tramos == tramo
        filas_tramos.append({"celda": etiqueta, "tramo": tramo, "turnos": int(m.sum()),
                             "f1_macro": f1_score(pred["y_true"][m], pred["y_pred"][m], average="macro")})
tabla_tramos = pd.DataFrame(filas_tramos)
pivote_tramos = tabla_tramos.pivot(index="tramo", columns="celda", values="f1_macro").loc[TRAMOS]
pivote_tramos.insert(0, "turnos", tabla_tramos.groupby("tramo")["turnos"].first().loc[TRAMOS])

In [36]:
# ============================================================
# Tabla de F1 macro por tramo
# ============================================================
display(pivote_tramos.round(4))

celda,turnos,GRU,LSTM,RNN simple
tramo,,,,
≤ 10,3870,0.7405,0.7447,0.7381
11–20,2866,0.6809,0.7001,0.6695
> 20,1333,0.6364,0.6191,0.5623


In [37]:
# ============================================================
# Gráfico de F1 macro por tramo
# ============================================================
fig = px.bar(tabla_tramos, x="tramo", y="f1_macro", color="celda", barmode="group", color_discrete_sequence=COLORES,
             title="F1 macro por largo del turno y tipo de celda",
             labels={"f1_macro": "F1 macro (validación)", "tramo": "tokens por turno"})
fig.show()

**Hallazgo:**

- En turnos de hasta 10 tokens las tres celdas empatan (F1 macro de 0,738 a 0,745).
- La diferencia crece con el largo: entre 11 y 20 tokens la LSTM supera a la RNN simple por 3,1 pp, y en turnos de más de 20 tokens (1.333 turnos, el 16,5 % de validación) las celdas con compuertas la superan por 5,7 pp (LSTM) y 7,4 pp (GRU).
- Es el comportamiento que predice el desvanecimiento del gradiente: la RNN simple pierde la señal del acto cuando el turno es largo. Es el argumento principal para preferir una celda con compuertas, aunque en el total la diferencia sea un empate.

## 10. Experimentos de entrenamiento: tasa de aprendizaje, tamaño del lote y épocas

Se estudia el impacto de **la tasa de aprendizaje, el tamaño del lote y las épocas**, cada uno en su propio experimento.

### 10.1 Tasa de aprendizaje

La tasa de aprendizaje es el tamaño del paso de Adam en cada actualización. Una tasa baja converge lento (más épocas para llegar a la mejor); una alta avanza rápido, pero puede oscilar o sobreajustar antes. El rango 0,0001–0,003 rodea el valor por defecto de Adam.

In [38]:
# ============================================================
# Experimento 2: learning_rate
# ============================================================
variantes_learning_rate = {
    "0.0001": dict(learning_rate=0.0001),
    "0.0005": dict(learning_rate=0.0005),
    "0.001": dict(learning_rate=0.001),
    "0.003": dict(learning_rate=0.003),
}
tabla_learning_rate = ejecutar_experimento("learning_rate", variantes_learning_rate, CONFIG_ACTUAL)

[learning_rate] 0.0001: guardado
[learning_rate] 0.0005: guardado
[learning_rate] 0.001: guardado
[learning_rate] 0.003: guardado


In [39]:
# ============================================================
# Resultados del experimento 2
# ============================================================
display(tabla_learning_rate[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,0.001,0.7154,0.7764,0.5886,0.0766,5,1310468,6.2263,0.0000,True,guardado
1,0.0005,0.7133,0.7779,0.5834,0.0549,5,1310468,6.5474,0.0021,True,guardado
2,0.0001,0.6972,0.7700,0.6039,0.0572,14,1310468,6.5745,0.0182,True,guardado
3,0.003,0.6908,0.7673,0.5932,0.0445,2,1310468,6.4735,0.0246,False,guardado


In [40]:
# ============================================================
# Selección de la variante del experimento 2
# ============================================================
elegida = seleccionar_variante(tabla_learning_rate)
# elegida = "0.001"  # Cambio Manual
aplicar_variante("learning_rate", variantes_learning_rate, elegida)

Mejor F1 macro de validación: 0.001
En empate técnico (dentro de 2.35 pp): 0.0005, 0.0001


,valor
celda,LSTM
units,64
capas,1
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:**

- La tasa de aprendizaje controla la velocidad de convergencia: con 0,0001 la mejor época es la 14; con 0,0005 y 0,001, la 5; con 0,003, la 2.
- 0,003 es la única variante fuera del rango de error (−2,46 pp): llega a su mejor punto en 2 épocas y sobreajusta enseguida. Su brecha es la menor (4,5 pp) solo porque el entrenamiento se detiene muy temprano.
- 0,001 y 0,0005 empatan (0,21 pp de diferencia, misma mejor época). Se mantiene 0,001.

### 10.2 Tamaño del lote

El tamaño del lote define cuántos turnos se usan en cada actualización. Con 87.170 turnos, los lotes de 32, 64, 128 y 256 producen ≈ 2.724, 1.362, 681 y 341 actualizaciones por época. Un lote grande es más rápido por época, pero actualiza menos veces y estima el gradiente con menos ruido.

In [41]:
# ============================================================
# Experimento 3: batch_size
# ============================================================
variantes_batch_size = {
    "32": dict(batch_size=32),
    "64": dict(batch_size=64),
    "128": dict(batch_size=128),
    "256": dict(batch_size=256),
}
tabla_batch_size = ejecutar_experimento("batch_size", variantes_batch_size, CONFIG_ACTUAL)
tabla_batch_size = ampliar_rango("batch_size", variantes_batch_size, CONFIG_ACTUAL, "batch_size",
                                 arriba=lambda v: v * 2, abajo=lambda v: v // 2, minimo=16, maximo=512)

[batch_size] 32: guardado
[batch_size] 64: guardado
[batch_size] 128: guardado
[batch_size] 256: guardado


In [42]:
# ============================================================
# Resultados del experimento 3
# ============================================================
display(tabla_batch_size[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,64,0.7154,0.7764,0.5886,0.0766,5,1310468,6.2263,0.0000,True,guardado
1,256,0.7045,0.7762,0.5981,0.0566,5,1310468,2.0645,0.0109,True,guardado
2,32,0.6983,0.7726,0.5834,0.0544,3,1310468,12.4238,0.0171,True,guardado
3,128,0.6974,0.7705,0.5892,0.0433,3,1310468,3.6375,0.0180,True,guardado


In [43]:
# ============================================================
# Selección de la variante del experimento 3
# ============================================================
elegida = seleccionar_variante(tabla_batch_size)
# elegida = "64"  # Cambio Manual
aplicar_variante("batch_size", variantes_batch_size, elegida)

Mejor F1 macro de validación: 64
En empate técnico (dentro de 2.35 pp): 256, 32, 128


,valor
celda,LSTM
units,64
capas,1
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:**

- Las cuatro variantes empatan (diferencias de hasta 1,80 pp). El tamaño del lote cambia sobre todo el costo: 12,4 s por época con 32, 6,6 s con 64, 3,6 s con 128 y 2,1 s con 256, seis veces más rápido que con 32.
- Se mantiene 64 por mayor F1. El lote de 256 es la alternativa eficiente (−1,09 pp y 3,2 veces más rápido); su combinación con una tasa mayor se verifica en la sección 17.2.

### 10.3 Épocas

En los experimentos las épocas las decide el Early Stopping. Para ver su efecto directo, se entrena la configuración actual **30 épocas sin detenerse** (`paciencia=0`) y se registra cada época: en qué época se minimiza la pérdida de validación, en cuál se maximiza el F1 y desde cuándo aparece el sobreajuste.

In [44]:
# ============================================================
# Experimento 4: entrenamiento sin Early Stopping
# ============================================================
_, historia_epocas, _ = entrenar_y_registrar(CONFIG_ACTUAL, epocas=30, paciencia=0, verbose=True, forzar=True)
tabla_epocas = pd.DataFrame(historia_epocas, index=range(1, len(historia_epocas["loss_train"]) + 1))
tabla_epocas["brecha_train_val"] = tabla_epocas["exactitud_train"] - tabla_epocas["exactitud_val"]
tabla_epocas.index.name = "epoca"
print(f"Mínima pérdida de validación en la época {int(tabla_epocas['loss_val'].idxmin())}; "
      f"máximo F1 macro en la época {int(tabla_epocas['f1_val'].idxmax())}.")

Época 01 | loss train 0.6304 | loss val 0.6259 | exactitud val 0.7536 | F1 macro val 0.6600
Época 02 | loss train 0.5191 | loss val 0.6026 | exactitud val 0.7661 | F1 macro val 0.6888
Época 03 | loss train 0.4724 | loss val 0.5910 | exactitud val 0.7725 | F1 macro val 0.6996
Época 04 | loss train 0.4315 | loss val 0.5975 | exactitud val 0.7804 | F1 macro val 0.7064
Época 05 | loss train 0.3926 | loss val 0.5886 | exactitud val 0.7764 | F1 macro val 0.7154
Época 06 | loss train 0.3553 | loss val 0.6197 | exactitud val 0.7767 | F1 macro val 0.7093
Época 07 | loss train 0.3201 | loss val 0.6402 | exactitud val 0.7797 | F1 macro val 0.7195
Época 08 | loss train 0.2882 | loss val 0.6932 | exactitud val 0.7717 | F1 macro val 0.7085
Época 09 | loss train 0.2586 | loss val 0.7310 | exactitud val 0.7748 | F1 macro val 0.7125
Época 10 | loss train 0.2328 | loss val 0.7788 | exactitud val 0.7680 | F1 macro val 0.7044
Época 11 | loss train 0.2107 | loss val 0.8063 | exactitud val 0.7712 | F1 macro

In [45]:
# ============================================================
# Métricas en épocas seleccionadas
# ============================================================
display(tabla_epocas.loc[[3, 5, 10, 15, 20, 30]].round(4))

,loss_train,exactitud_train,loss_val,exactitud_val,f1_val,brecha_train_val
epoca,,,,,,
3,0.4724,0.8218,0.5910,0.7725,0.6996,0.0493
5,0.3926,0.8531,0.5886,0.7764,0.7154,0.0766
10,0.2328,0.9138,0.7788,0.7680,0.7044,0.1458
15,0.1481,0.9429,0.9957,0.7645,0.7026,0.1784
20,0.1117,0.9547,1.1406,0.7612,0.7079,0.1935
30,0.0865,0.9619,1.3051,0.7621,0.6981,0.1999


In [46]:
# ============================================================
# Curvas sin Early Stopping
# ============================================================
graficar_curvas(historia_epocas, "Configuración actual entrenada 30 épocas sin Early Stopping")

**Hallazgo:**

- La pérdida de validación es mínima en la época 5 (0,589) y desde ahí sube sin parar, hasta 1,305 en la época 30, mientras la exactitud de entrenamiento llega a 0,962. La brecha pasa de 7,7 pp a 20,0 pp: sobreajuste claro.
- El F1 macro, en cambio, casi no cae: entre las épocas 5 y 30 oscila entre 0,696 y 0,720 (máximo de 0,7195 en la época 7). El modelo se vuelve cada vez más seguro de sus errores (la pérdida crece) sin cambiar mucho cuáles turnos acierta.
- Con la mejor época cerca de la 5, el tope de 25 épocas con paciencia 3 deja margen de sobra: ninguna corrida del notebook llegó al tope todavía mejorando.

## 11. Experimentos de arquitectura

Se ajustan la capacidad de la red y el preprocesamiento, siempre de a un hiperparámetro. Si la mejor variante queda en el borde del rango, `ampliar_rango` agrega el valor siguiente.

### 11.1 Unidades del estado oculto (`units`)

Tamaño del estado hₜ, la «memoria» de la red. Más unidades dan más capacidad, pero más parámetros y riesgo de sobreajuste.

In [47]:
# ============================================================
# Experimento 5: units
# ============================================================
variantes_units = {
    "32": dict(units=32),
    "64": dict(units=64),
    "128": dict(units=128),
    "256": dict(units=256),
}
tabla_units = ejecutar_experimento("units", variantes_units, CONFIG_ACTUAL)
tabla_units = ampliar_rango("units", variantes_units, CONFIG_ACTUAL, "units",
                            arriba=lambda v: v * 2, abajo=lambda v: v // 2, minimo=16, maximo=512)

[units] 32: guardado
[units] 64: guardado
[units] 128: guardado
[units] 256: guardado


In [48]:
# ============================================================
# Resultados del experimento 5
# ============================================================
display(tabla_units[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,64,0.7154,0.7764,0.5886,0.0766,5,1310468,6.2263,0.0000,True,guardado
1,128,0.7076,0.7780,0.5851,0.0478,3,1393156,6.5731,0.0078,True,guardado
2,256,0.7069,0.7804,0.5727,0.0470,3,1656836,6.8390,0.0085,True,guardado
3,32,0.6945,0.7682,0.5983,0.0618,4,1281412,6.7705,0.0209,True,guardado


In [49]:
# ============================================================
# Selección de la variante del experimento 5
# ============================================================
elegida = seleccionar_variante(tabla_units)
# elegida = "128"  # Cambio Manual
aplicar_variante("units", variantes_units, elegida)

Mejor F1 macro de validación: 64
En empate técnico (dentro de 2.35 pp): 128, 256, 32


,valor
celda,LSTM
units,64
capas,1
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:**

- Las cuatro variantes empatan (diferencias de hasta 2,09 pp). Con 64 unidades se obtiene el mayor F1.
- 128 y 256 unidades bajan la brecha (4,8 y 4,7 pp, contra 7,7 pp) y la pérdida (256 tiene la menor, 0,573), pero no mejoran el F1.
- El mejor valor queda en el interior del rango probado (32–256), por lo que no se amplía. Se mantiene 64 por mayor F1 y menor costo; la combinación de más unidades con Dropout se revisa en la sección 17.3.

### 11.2 Número de capas recurrentes (`capas`)

Apilar capas permite representar patrones más complejos; las capas intermedias entregan la secuencia completa a la siguiente. Se parte con 1 y 2 capas y se amplía si la mejor queda en el borde.

In [50]:
# ============================================================
# Experimento 6: capas
# ============================================================
variantes_capas = {
    "1": dict(capas=1),
    "2": dict(capas=2),
}
tabla_capas = ejecutar_experimento("capas", variantes_capas, CONFIG_ACTUAL)
tabla_capas = ampliar_rango("capas", variantes_capas, CONFIG_ACTUAL, "capas",
                            arriba=lambda v: v + 1, abajo=lambda v: v - 1, minimo=1, maximo=4)

[capas] 1: guardado
[capas] 2: guardado
[capas] La mejor variante está en el borde del rango: se agrega capas = 3
[capas] 3: guardado
[capas] La mejor variante está en el borde del rango: se agrega capas = 4
[capas] 4: guardado


In [51]:
# ============================================================
# Resultados del experimento 6
# ============================================================
display(tabla_capas[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,3,0.7243,0.7814,0.5799,0.0673,5,1377028,8.8663,0.0000,True,guardado
1,4,0.7219,0.7784,0.5780,0.0701,5,1410308,11.1864,0.0024,True,guardado
2,2,0.7179,0.7768,0.5846,0.0750,5,1343748,7.9970,0.0064,True,guardado
3,1,0.7154,0.7764,0.5886,0.0766,5,1310468,6.2263,0.0089,True,guardado


In [52]:
# ============================================================
# Selección de la variante del experimento 6
# ============================================================
elegida = seleccionar_variante(tabla_capas)
# elegida = "1"  # Cambio Manual
aplicar_variante("capas", variantes_capas, elegida)

Mejor F1 macro de validación: 3
En empate técnico (dentro de 2.35 pp): 4, 2, 1


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Hallazgo:**

- Como la mejor variante inicial (2 capas) quedaba en el borde del rango, se agregaron 3 y 4 capas. 3 capas obtiene el mayor F1 (0,7243) y 4 baja a 0,7219: el óptimo queda en el interior.
- Las cuatro variantes empatan (rango de 0,89 pp). 3 capas cuesta un 33 % más de tiempo por época que 1 capa (8,8 s contra 6,6 s), con apenas un 5 % más de parámetros. Se elige 3 por mayor F1.
- Las decisiones de los experimentos siguientes (11.3 a 12.2) se tomaron con 3 capas. Antes de ampliar el rango se habían tomado con 2 capas, y algunas resultaron distintas: la sección 17.5 compara ambas cadenas.

### 11.3 Lectura bidireccional (`bidireccional`)

La red lee el turno en ambos sentidos. Es legítimo porque se clasifica el turno completo; no se podría usar para generar texto token a token.

In [53]:
# ============================================================
# Experimento 7: bidireccional
# ============================================================
variantes_bidireccional = {
    "unidireccional": dict(bidireccional=False),
    "bidireccional": dict(bidireccional=True),
}
tabla_bidireccional = ejecutar_experimento("bidireccional", variantes_bidireccional, CONFIG_ACTUAL)

[bidireccional] unidireccional: guardado
[bidireccional] bidireccional: guardado


In [54]:
# ============================================================
# Resultados del experimento 7
# ============================================================
display(tabla_bidireccional[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,unidireccional,0.7243,0.7814,0.5799,0.0673,5,1377028,8.8663,0.0000,True,guardado
1,bidireccional,0.7059,0.7792,0.5681,0.0476,3,1559044,12.2800,0.0184,True,guardado


In [55]:
# ============================================================
# Selección de la variante del experimento 7
# ============================================================
elegida = seleccionar_variante(tabla_bidireccional)
# elegida = "bidireccional"  # Cambio Manual
aplicar_variante("bidireccional", variantes_bidireccional, elegida)

Mejor F1 macro de validación: unidireccional
En empate técnico (dentro de 2.35 pp): bidireccional


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:** la lectura bidireccional queda 1,84 pp bajo la unidireccional (empate), con un 13 % más de parámetros y un 37 % más de tiempo por época (12,3 s contra 9,0 s). Tiene menor pérdida (0,568) y menor brecha (4,8 pp), pero no mejora el F1. Se mantiene la lectura unidireccional, porque el costo adicional no se refleja en el criterio principal.

### 11.4 Dimensión del embedding (`embed_dim`)

Cantidad de valores que describen cada token.

In [56]:
# ============================================================
# Experimento 8: embed_dim
# ============================================================
variantes_embed_dim = {
    "64": dict(embed_dim=64),
    "128": dict(embed_dim=128),
    "256": dict(embed_dim=256),
}
tabla_embed_dim = ejecutar_experimento("embed_dim", variantes_embed_dim, CONFIG_ACTUAL)
tabla_embed_dim = ampliar_rango("embed_dim", variantes_embed_dim, CONFIG_ACTUAL, "embed_dim",
                                arriba=lambda v: v * 2, abajo=lambda v: v // 2, minimo=16, maximo=512)

[embed_dim] 64: guardado
[embed_dim] 128: guardado
[embed_dim] 256: guardado


In [57]:
# ============================================================
# Resultados del experimento 8
# ============================================================
display(tabla_embed_dim[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,128,0.7243,0.7814,0.5799,0.0673,5,1377028,8.8663,0.0000,True,guardado
1,64,0.7220,0.7749,0.5800,0.0594,5,730372,8.7250,0.0023,True,guardado
2,256,0.7113,0.7821,0.5726,0.0481,3,2670340,9.5550,0.0130,True,guardado


In [58]:
# ============================================================
# Selección de la variante del experimento 8
# ============================================================
elegida = seleccionar_variante(tabla_embed_dim)
# elegida = "128"  # Cambio Manual
aplicar_variante("embed_dim", variantes_embed_dim, elegida)

Mejor F1 macro de validación: 128
En empate técnico (dentro de 2.35 pp): 64, 256


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:**

- 128 dimensiones obtiene el mayor F1 (0,7243). 64 queda 0,23 pp abajo con la mitad de parámetros (0,73 M contra 1,38 M), y 256 queda 1,30 pp abajo con el doble.
- Las tres empatan y el mejor valor está en el interior del rango, por lo que no se amplía. Se mantiene 128; 64 sería la alternativa liviana con un F1 prácticamente igual.

### 11.5 Largo máximo del turno (`maxlen`)

Con 20, 30 y 40 tokens se cubre ≈ 83,3 %, 93,7 % y 97,5 % de los turnos completos (sección 3.5). No se prueban valores mayores: por sobre 40 tokens el beneficio posible es solo el 2,5 % restante de turnos, que se truncan conservando su inicio.

In [59]:
# ============================================================
# Experimento 9: maxlen
# ============================================================
variantes_maxlen = {
    "20": dict(maxlen=20),
    "30": dict(maxlen=30),
    "40": dict(maxlen=40),
}
tabla_maxlen = ejecutar_experimento("maxlen", variantes_maxlen, CONFIG_ACTUAL)

[maxlen] 20: guardado
[maxlen] 30: guardado
[maxlen] 40: guardado


In [60]:
# ============================================================
# Resultados del experimento 9
# ============================================================
display(tabla_maxlen[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,40,0.7243,0.7814,0.5799,0.0673,5,1377028,8.8663,0.0000,True,guardado
1,30,0.7045,0.7777,0.5866,0.0560,4,1377028,8.6709,0.0198,True,guardado
2,20,0.6859,0.7634,0.6169,0.0469,3,1377028,8.5165,0.0384,False,guardado


In [61]:
# ============================================================
# Selección de la variante del experimento 9
# ============================================================
elegida = seleccionar_variante(tabla_maxlen)
# elegida = "40"  # Cambio Manual
aplicar_variante("maxlen", variantes_maxlen, elegida)

Mejor F1 macro de validación: 40
En empate técnico (dentro de 2.35 pp): 30


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Hallazgo:** `maxlen` es la segunda decisión que supera el rango de error: con 20 tokens el F1 macro cae 3,84 pp (0,6859) y con 30, 1,98 pp. Aunque solo el 2,5 % de los turnos supera los 40 tokens, recortar a 20 trunca el 16,7 % de los turnos y les quita información útil para clasificarlos. Se mantiene 40, que es además el límite justificado en el Markdown de esta sección.

### 11.6 Tamaño del vocabulario (`min_freq`)

Con `min_freq` 1, 2 y 3 el vocabulario tiene ≈ 22.000, 13.400 y 9.900 tokens. `min_freq = 1` es el límite natural (todo el vocabulario de `train`). Un vocabulario chico aumenta los `<unk>`; uno grande agrega palabras raras que casi no se entrenan y más parámetros en el embedding.

In [62]:
# ============================================================
# Experimento 10: min_freq
# ============================================================
variantes_min_freq = {
    "1": dict(min_freq=1),
    "2": dict(min_freq=2),
    "3": dict(min_freq=3),
}
tabla_min_freq = ejecutar_experimento("min_freq", variantes_min_freq, CONFIG_ACTUAL)

[min_freq] 1: guardado
[min_freq] 2: guardado
[min_freq] 3: guardado


In [63]:
# ============================================================
# Resultados del experimento 10
# ============================================================
display(tabla_min_freq[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,3,0.7243,0.7814,0.5799,0.0673,5,1377028,8.8663,0.0000,True,guardado
1,2,0.7040,0.7793,0.5764,0.0407,3,1828228,8.9045,0.0203,True,guardado
2,1,0.7036,0.7754,0.5819,0.0465,3,2942340,9.4239,0.0207,True,guardado


In [64]:
# ============================================================
# Selección de la variante del experimento 10
# ============================================================
elegida = seleccionar_variante(tabla_min_freq)
# elegida = "3"  # Cambio Manual
aplicar_variante("min_freq", variantes_min_freq, elegida)

Mejor F1 macro de validación: 3
En empate técnico (dentro de 2.35 pp): 2, 1


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Hallazgo:**

- Con 3 capas, `min_freq = 3` obtiene el mayor F1 (0,7243), unos 2,0 pp sobre 2 y 1, casi en el límite del rango de error, con la mitad o un tercio de los parámetros (1,38 M contra 1,83 M y 2,94 M).
- Con la cadena de 2 capas, anterior a la ampliación del experimento 11.2, se había elegido `min_freq = 1` (sección 17.5). Es un ejemplo concreto de dependencia del orden: cambiar una decisión anterior cambió qué vocabulario conviene.

## 12. Experimentos de regularización y desbalance

Se prueban el Dropout y los pesos de clase.

### 12.1 Dropout

Apaga al azar una fracción de las activaciones antes de la capa de salida (y entre capas recurrentes si hay más de una), solo durante el entrenamiento. Su objetivo es reducir la brecha entre entrenamiento y validación. 0,0 es el límite natural inferior.

In [65]:
# ============================================================
# Experimento 11: dropout
# ============================================================
variantes_dropout = {
    "0.0": dict(dropout=0.0),
    "0.2": dict(dropout=0.2),
    "0.4": dict(dropout=0.4),
}
tabla_dropout = ejecutar_experimento("dropout", variantes_dropout, CONFIG_ACTUAL)
tabla_dropout = ampliar_rango("dropout", variantes_dropout, CONFIG_ACTUAL, "dropout",
                              arriba=lambda v: round(v + 0.2, 1), abajo=lambda v: round(v - 0.2, 1), minimo=0.0, maximo=0.6)

[dropout] 0.0: guardado
[dropout] 0.2: guardado
[dropout] 0.4: guardado


In [66]:
# ============================================================
# Resultados del experimento 11
# ============================================================
display(tabla_dropout[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,0.0,0.7243,0.7814,0.5799,0.0673,5,1377028,8.8663,0.0000,True,guardado
1,0.4,0.7153,0.7792,0.5823,0.0541,5,1377028,9.0047,0.0090,True,guardado
2,0.2,0.7146,0.7783,0.5776,0.0635,5,1377028,9.0151,0.0097,True,guardado


In [67]:
# ============================================================
# Selección de la variante del experimento 11
# ============================================================
elegida = seleccionar_variante(tabla_dropout)
# elegida = "0.2"  # Cambio Manual
aplicar_variante("dropout", variantes_dropout, elegida)

Mejor F1 macro de validación: 0.0
En empate técnico (dentro de 2.35 pp): 0.4, 0.2


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:** Dropout 0,2 y 0,4 reducen la brecha (6,4 y 5,4 pp, contra 6,7 pp), pero bajan el F1 entre 0,9 y 1,0 pp (empate). Como el Early Stopping ya detiene el entrenamiento cerca de la época 5, antes de que el sobreajuste crezca, el Dropout aporta poco. Se mantiene 0,0, que es el límite natural del rango.

### 12.2 Pesos de clase

Con pesos balanceados, un error en *commissive* (9,3 % de los turnos) cuesta en la pérdida unas 2,7 veces más que uno en *inform*. Se espera que suba la exhaustividad de las clases pequeñas, a cambio de algo de exactitud global. Se entrenan siempre, porque sus modelos se usan para el análisis por clase.

In [68]:
# ============================================================
# Experimento 12: pesos de clase
# ============================================================
variantes_pesos_clase = {
    "sin pesos": dict(pesos_clase=False),
    "balanceados": dict(pesos_clase=True),
}
tabla_pesos_clase = ejecutar_experimento("pesos_clase", variantes_pesos_clase, CONFIG_ACTUAL, forzar=True)
filas_por_clase = []
for etiqueta, (config, modelo) in modelos_ultimo_experimento.items():
    vocab, _, datos_val, _ = preparar_datos(config["maxlen"], config["min_freq"])
    pred = predecir(modelo, crear_loader_eval(datos_val, vocab["<pad>"]))
    precision, exhaustividad, f1, _ = precision_recall_fscore_support(pred["y_true"], pred["y_pred"], zero_division=0)
    for acto, p, e, f in zip(ACTOS, precision, exhaustividad, f1):
        filas_por_clase.append({"variante": etiqueta, "acto": acto, "precision": p, "exhaustividad": e, "f1": f})
tabla_pesos_por_clase = pd.DataFrame(filas_por_clase).pivot(index="acto", columns="variante").loc[ACTOS]

[pesos_clase] sin pesos: entrenado
[pesos_clase] balanceados: entrenado


In [69]:
# ============================================================
# Resultados del experimento 12
# ============================================================
display(tabla_pesos_clase[COLUMNAS_TABLA].round(4))

,variante,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_mejor,dentro_del_margen,origen
0,sin pesos,0.7243,0.7814,0.5799,0.0673,5,1377028,8.9014,0.0000,True,entrenado
1,balanceados,0.7178,0.7508,0.6214,0.0353,4,1377028,8.9096,0.0065,True,entrenado


In [70]:
# ============================================================
# Precisión, exhaustividad y F1 por clase según los pesos
# ============================================================
display(tabla_pesos_por_clase.round(4))

precision           exhaustividad                    f1          
variante   balanceados sin pesos   balanceados sin pesos balanceados sin pesos
acto                                                                          
inform          0.8261    0.7797        0.6858    0.8221      0.7494    0.8003
question        0.9069    0.8988        0.9287    0.9376      0.9177    0.9178
directive       0.6912    0.7180        0.7048    0.7087      0.6979    0.7134
commissive      0.4243    0.5492        0.6270    0.4043      0.5061    0.4658

In [71]:
# ============================================================
# Selección de la variante del experimento 12
# ============================================================
elegida = seleccionar_variante(tabla_pesos_clase)
elegida = "balanceados"  # Cambio Manual: empate técnico; se prioriza la exhaustividad de commissive
aplicar_variante("pesos_clase", variantes_pesos_clase, elegida)

Mejor F1 macro de validación: sin pesos
En empate técnico (dentro de 2.35 pp): balanceados


,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


**Observación:**

- Sin pesos se obtiene el mayor F1 macro (0,7243); con pesos balanceados, 0,7178. La diferencia (0,65 pp) está dentro del rango de error: empate técnico.
- Con pesos balanceados, la exhaustividad de *commissive* sube de 0,404 a 0,627 y su F1 de 0,466 a 0,506 (+4,0 pp). Como contrapartida, la exhaustividad de *inform* baja de 0,822 a 0,686 y la exactitud global de 0,781 a 0,751.

**Decisión por contexto (`# Cambio Manual`):** se eligen los **pesos balanceados**. Hay empate en el criterio principal, *commissive* es la clase más débil y el F1 macro busca que las cuatro clases pesen igual; los pesos mejoran la clase minoritaria sin salir del rango de error. La sección 17.4 respalda la decisión con 5 semillas.

## 13. Modelo final

### 13.1 Configuración final

La configuración final es `CONFIG_ACTUAL` después de todos los experimentos.

In [72]:
# ============================================================
# Configuración final
# ============================================================
CONFIG_FINAL = dict(CONFIG_ACTUAL)
mostrar_config(CONFIG_FINAL)

,valor
celda,LSTM
units,64
capas,3
bidireccional,False
embed_dim,128
dropout,0.0
learning_rate,0.001
batch_size,64
maxlen,40
min_freq,3


In [73]:
# ============================================================
# Decisiones tomadas
# ============================================================
display(pd.DataFrame({"experimento": list(decisiones), "variante_elegida": list(decisiones.values())}))

,experimento,variante_elegida
0,celda,LSTM
1,learning_rate,0.001
2,batch_size,64
3,units,64
4,capas,3
5,bidireccional,unidireccional
6,embed_dim,128
7,maxlen,40
8,min_freq,3
9,dropout,0.0


### 13.2 Entrenamiento del modelo final

El modelo final se entrena con **el mismo régimen que todos los experimentos** (máximo 25 épocas, paciencia 3) y la semilla principal, para que su resultado sea directamente comparable con el de la selección. El experimento 10.3 muestra que la pérdida de validación es mínima alrededor de la época 5 y luego sube, por lo que ese régimen alcanza para que el modelo llegue a su mejor época. Se entrena siempre, porque se necesitan sus pesos.

In [74]:
# ============================================================
# Entrenamiento del modelo final
# ============================================================
fila_final, historia_final, modelo_final = entrenar_y_registrar(CONFIG_FINAL, verbose=True, forzar=True)
print(modelo_final)

Época 01 | loss train 0.7861 | loss val 0.6812 | exactitud val 0.7086 | F1 macro val 0.6796
Época 02 | loss train 0.6554 | loss val 0.6488 | exactitud val 0.7412 | F1 macro val 0.7064
Época 03 | loss train 0.5971 | loss val 0.6329 | exactitud val 0.7423 | F1 macro val 0.7122
Época 04 | loss train 0.5430 | loss val 0.6214 | exactitud val 0.7508 | F1 macro val 0.7178
Época 05 | loss train 0.4868 | loss val 0.6731 | exactitud val 0.7332 | F1 macro val 0.7062
Época 06 | loss train 0.4305 | loss val 0.6513 | exactitud val 0.7580 | F1 macro val 0.7242
Época 07 | loss train 0.3796 | loss val 0.6971 | exactitud val 0.7490 | F1 macro val 0.7180
ClasificadorRecurrente(
  (embedding): Embedding(9848, 128, padding_idx=1)
  (rnn): LSTM(128, 64, num_layers=3, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=64, out_features=4, bias=True)
)


In [75]:
# ============================================================
# Métricas del modelo final
# ============================================================
display(pd.DataFrame([fila_final]).round(4))

,exactitud_train,exactitud_val,f1_macro_val,loss_val,brecha_train_val,mejor_epoca,epocas_entrenadas,parametros,segundos,segundos_por_epoca,origen
0,0.7861,0.7508,0.7178,0.6214,0.0353,4,7,1377028,62.7615,8.9659,entrenado


### 13.3 Curvas de entrenamiento del modelo final

Gráfico de curvas del modelo elegido; el resto de las configuraciones se compara en tablas.

In [76]:
# ============================================================
# Curvas del modelo final
# ============================================================
graficar_curvas(historia_final, f"Modelo final ({CONFIG_FINAL['celda']})")

**Observación:**

- El modelo final repite exactamente su corrida de la selección (F1 macro 0,7178, exactitud 0,7508, mejor época 4), lo que confirma la reproducibilidad con la misma configuración y semilla. Se detiene en la época 7 por la paciencia de 3 épocas.
- Su brecha es de 3,5 pp, la menor del notebook: los pesos de clase actúan también como regularización.

**Hallazgo:** en la época 6 el F1 macro llega a 0,724, pero el Early Stopping monitorea la pérdida de validación, que fue mínima en la época 4. Pérdida y F1 no siempre eligen la misma época.

## 14. Evaluación del modelo final en el conjunto de test

El conjunto de test se usa **una sola vez**, aquí. Qué mide cada métrica en este problema:

- **Exactitud:** proporción de turnos cuyo acto se predijo bien.
- **Precisión de una clase** (por ejemplo, *question*): de los turnos que el modelo marcó como pregunta, cuántos lo eran. Una precisión baja indica falsas alarmas.
- **Exhaustividad de una clase:** de las preguntas reales, cuántas detectó. Una exhaustividad baja indica preguntas que se le escapan.
- **F1:** media armónica entre precisión y exhaustividad; solo es alto si ambas lo son.
- **Promedio macro** (todas las clases pesan igual) frente a **ponderado** (cada clase pesa según su cantidad de turnos): si el macro es mucho menor que el ponderado, al modelo le van mal las clases pequeñas.

### 14.1 Métricas globales

Se compara el modelo final con las dos líneas base, ahora en test.

In [77]:
# ============================================================
# Métricas globales en test
# ============================================================
vocab_final, _, _, datos_test = preparar_datos(CONFIG_FINAL["maxlen"], CONFIG_FINAL["min_freq"])
pred_test = predecir(modelo_final, crear_loader_eval(datos_test, vocab_final["<pad>"]))
y_test, y_pred_test = pred_test["y_true"], pred_test["y_pred"]

tabla_test = pd.DataFrame([
    {"modelo": "Línea base: clase mayoritaria", **metricas(y_test, linea_base_mayoritaria(turnos_test))},
    {"modelo": "Línea base: regla «?»", **metricas(y_test, linea_base_pregunta(turnos_test))},
    {"modelo": f"Modelo final ({CONFIG_FINAL['celda']})", **metricas(y_test, y_pred_test)},
]).round(4)
display(tabla_test)

,modelo,exactitud,precision_macro,exhaustividad_macro,f1_macro,f1_ponderado
0,Línea base: clase mayoritaria,0.4566,0.1141,0.2500,0.1567,0.2862
1,Línea base: regla «?»,0.7284,0.3715,0.4911,0.4226,0.6215
2,Modelo final (LSTM),0.7733,0.7053,0.7467,0.7162,0.7840


**Observación:**

- En test, el modelo final obtiene una exactitud de 0,773 y un F1 macro de 0,716. Frente a la regla «?» (0,728 y 0,423), son 4,5 pp más de exactitud y 29,4 pp más de F1 macro.
- La exactitud de test supera a la de validación (0,773 contra 0,751), como se anticipó en la sección 3.4: test tiene menos turnos *directive* y *commissive*.
- El F1 ponderado (0,784) supera al macro (0,716): al modelo le van peor las clases pequeñas.

### 14.2 Métricas por clase

Precisión, exhaustividad y F1 de cada acto, en test y también en validación (más errores por clase, evidencia más sólida).

In [78]:
# ============================================================
# Reporte por clase en test
# ============================================================
reporte = pd.DataFrame(classification_report(y_test, y_pred_test, target_names=ACTOS, output_dict=True, zero_division=0)).T
display(reporte.round(4))

,precision,recall,f1-score,support
inform,0.8888,0.7326,0.8032,3534.0000
question,0.9170,0.9299,0.9234,2210.0000
directive,0.6344,0.6964,0.6639,1278.0000
commissive,0.3812,0.6281,0.4745,718.0000
accuracy,0.7733,0.7733,0.7733,0.7733
macro avg,0.7053,0.7467,0.7162,7740.0000
weighted avg,0.8077,0.7733,0.7840,7740.0000


In [79]:
# ============================================================
# Reporte por clase en validación
# ============================================================
_, _, datos_val_final, _ = preparar_datos(CONFIG_FINAL["maxlen"], CONFIG_FINAL["min_freq"])
pred_val_final = predecir(modelo_final, crear_loader_eval(datos_val_final, vocab_final["<pad>"]))
reporte_val = pd.DataFrame(classification_report(pred_val_final["y_true"], pred_val_final["y_pred"],
                                                 target_names=ACTOS, output_dict=True, zero_division=0)).T
display(reporte_val.round(4))

,precision,recall,f1-score,support
inform,0.8261,0.6858,0.7494,3125.0000
question,0.9069,0.9287,0.9177,2244.0000
directive,0.6912,0.7048,0.6979,1775.0000
commissive,0.4243,0.6270,0.5061,925.0000
accuracy,0.7508,0.7508,0.7508,0.7508
macro avg,0.7121,0.7366,0.7178,8069.0000
weighted avg,0.7728,0.7508,0.7570,8069.0000


In [80]:
# ============================================================
# Gráfico de métricas por clase en test
# ============================================================
por_clase = (reporte.loc[ACTOS, ["precision", "recall", "f1-score"]]
             .rename(columns={"precision": "precisión", "recall": "exhaustividad", "f1-score": "F1"})
             .reset_index(names="acto").melt(id_vars="acto", var_name="metrica", value_name="valor"))
fig = px.bar(por_clase, x="acto", y="valor", color="metrica", barmode="group", range_y=[0, 1],
             color_discrete_sequence=COLORES[::-1], title="Métricas por acto en test", labels={"metrica": "métrica"})
fig.show()

**Hallazgo:**

- *question* es la clase mejor resuelta (F1 de 0,92 en test y en validación) y *commissive* la peor (0,47 en test y 0,51 en validación).
- En *commissive* el problema es de **precisión**: su exhaustividad es 0,63, pero su precisión es solo 0,38 en test (0,42 en validación). El modelo la **sobrepredice**, efecto directo de su peso de clase (2,7). En *inform* ocurre lo contrario: precisión alta (0,89) y exhaustividad menor (0,73).
- Validación, con más errores por clase, muestra el mismo patrón, así que la conclusión no depende solo de test.

### 14.3 Matriz de confusión

Matriz normalizada por fila: cada fila muestra cómo se repartieron las predicciones de los turnos de un acto real. Luego, los pares más confundidos con sus conteos.

In [81]:
# ============================================================
# Matriz de confusión normalizada
# ============================================================
matriz = confusion_matrix(y_test, y_pred_test, normalize="true")
fig = px.imshow(matriz, x=ACTOS, y=ACTOS, text_auto=".2f", color_continuous_scale="Blues", zmin=0, zmax=1,
                labels={"x": "acto predicho", "y": "acto real", "color": "proporción"},
                title="Matriz de confusión normalizada por fila (test)")
fig.update_layout(width=600, height=520)
fig.show()

In [82]:
# ============================================================
# Pares de actos más confundidos (conteos)
# ============================================================
pares = pd.crosstab(pd.Series(y_test, name="real").map(dict(enumerate(ACTOS))),
                    pd.Series(y_pred_test, name="predicho").map(dict(enumerate(ACTOS))))
errores_pares = pares.stack().reset_index(name="turnos")
display(errores_pares[errores_pares["real"] != errores_pares["predicho"]].sort_values("turnos", ascending=False).head(5))

,real,predicho,turnos
8,inform,commissive,562
9,inform,directive,313
2,commissive,inform,165
4,directive,commissive,153
6,directive,inform,141


**Hallazgo:**

- La confusión principal es *inform* → *commissive* (562 turnos, el 15,9 % de los *inform* de test), seguida de *inform* → *directive* (313). La confusión anticipada en la sección 3.6 (*directive* → *question*) no aparece entre las cinco principales.
- El modelo distingue bien las preguntas; lo que mezcla son los actos sin una marca formal (*inform*, *directive* y *commissive*). La sobrepredicción de *commissive* concentra el costo de los pesos de clase.

### 14.4 Análisis de errores

Se revisan turnos mal clasificados y se separan los errores que **también** comete la regla «?» (difíciles por la forma del turno) de los que la regla sí acierta.

In [83]:
# ============================================================
# Análisis de errores en test
# ============================================================
analisis = turnos_test[["texto", "n_tokens"]].copy()
analisis["real"] = pd.Series(y_test).map(dict(enumerate(ACTOS))).values
analisis["predicho"] = pd.Series(y_pred_test).map(dict(enumerate(ACTOS))).values
analisis["probabilidad"] = pred_test["probas"].max(axis=1)
analisis["regla_acierta"] = linea_base_pregunta(turnos_test) == y_test
errores = analisis[analisis["real"] != analisis["predicho"]]
print(f"Errores: {len(errores):,} de {len(analisis):,} turnos")
print(f"Errores en los que la regla «?» también falla: {(~errores['regla_acierta']).mean():.1%}")
display(errores.sample(10, random_state=SEED).round(3))

Errores: 1,755 de 7,740 turnos
Errores en los que la regla «?» también falla: 43.0%


,texto,n_tokens,real,predicho,probabilidad,regla_acierta
1320,i am looking for a pair of shoes .,9,inform,directive,0.813,True
1648,please hurry .,3,commissive,directive,0.779,False
1634,"thank you . coffee , please .",7,directive,commissive,0.622,False
1891,you really should go to rachel's pet store .,9,inform,directive,0.812,True
4605,"office supplies , inc . robert speaking , may i help you ?",13,question,directive,0.644,True
5140,"oh , thanks , professor donald . this is really very nice of you .",15,inform,commissive,0.661,True
4939,i am so glad i caught you at home . i need your help !,15,inform,directive,0.788,True
3291,"there are two of us , but we only need a one-bedroom apartment .",14,inform,directive,0.526,True
5386,i'll say the movie's starting in the minute .,9,inform,commissive,0.666,True
4655,"yes , i do , susan . just a moment . here it is .",15,directive,commissive,0.806,False


**Hallazgo:**

- Hay 1.755 errores (22,7 % de los turnos de test). En el 43 % también falla la regla «?»; el 57 % restante son turnos que la regla sí acierta, sobre todo *inform* clasificados como *directive* o *commissive*.
- Muchos errores son turnos cuya función depende del diálogo y no del turno aislado: «i am looking for a pair of shoes .» (*inform* → *directive*) o «oh , thanks , professor donald . this is really very nice of you .» (*inform* → *commissive*). Otros parecen ambigüedad de la etiqueta, como «please hurry .», etiquetado como *commissive*.
- Estos errores sugieren que el turno aislado no basta: incluir también el turno anterior como entrada podría ayudar a resolverlos. Se plantea como mejora en la sección 18.1.

## 15. Comparación entre la configuración base y la final

Resume qué cambió desde `CONFIG_BASE` y cuánto aportó cada decisión. Una decisión **supera el margen** si la diferencia entre la mejor y la peor variante de su experimento es mayor que `MARGEN_ERROR`.

La comparación de esta sección usa una sola ejecución del modelo final, elegido entre muchas variantes evaluadas sobre la misma validación, frente a la media de 5 semillas del base: es una **cota optimista**. La comparación justa, con 5 semillas en ambos, está en la sección 17.4.

In [84]:
# ============================================================
# Configuración base vs. final
# ============================================================
config_base_vs_final = pd.DataFrame({"base": pd.Series(CONFIG_BASE).astype(str), "final": pd.Series(CONFIG_FINAL).astype(str)})
config_base_vs_final["cambio"] = config_base_vs_final["base"] != config_base_vs_final["final"]
display(config_base_vs_final)

,base,final,cambio
celda,RNN,LSTM,True
units,64,64,False
capas,1,3,True
bidireccional,False,False,False
embed_dim,128,128,False
dropout,0.0,0.0,False
learning_rate,0.001,0.001,False
batch_size,64,64,False
maxlen,40,40,False
min_freq,3,3,False


In [85]:
# ============================================================
# Aporte de cada decisión
# ============================================================
filas_decision = []
for tabla in resultados_experimentos:
    nombre = tabla.loc[0, "experimento"]
    if nombre in decisiones:
        rango = tabla["f1_macro_val"].max() - tabla["f1_macro_val"].min()
        f1_elegida = tabla.loc[tabla["variante"] == decisiones[nombre], "f1_macro_val"].iloc[0]
        filas_decision.append({"experimento": nombre, "variante_elegida": decisiones[nombre], "f1_elegida": f1_elegida,
                               "mejor_variante": tabla.loc[0, "variante"], "rango_entre_variantes": rango,
                               "supera_margen": rango > MARGEN_ERROR})
tabla_decisiones = pd.DataFrame(filas_decision)
print(f"F1 macro de validación: base {media_f1_base:.4f} (media de 5 semillas) → final {fila_final['f1_macro_val']:.4f} "
      f"(una ejecución; {(fila_final['f1_macro_val'] - media_f1_base) * 100:+.2f} pp)")
display(tabla_decisiones.round(4))

F1 macro de validación: base 0.6856 (media de 5 semillas) → final 0.7178 (una ejecución; +3.22 pp)


,experimento,variante_elegida,f1_elegida,mejor_variante,rango_entre_variantes,supera_margen
0,celda,LSTM,0.7154,LSTM,0.0179,False
1,learning_rate,0.001,0.7154,0.001,0.0246,True
2,batch_size,64,0.7154,64,0.0180,False
3,units,64,0.7154,64,0.0209,False
4,capas,3,0.7243,3,0.0089,False
5,bidireccional,unidireccional,0.7243,unidireccional,0.0184,False
6,embed_dim,128,0.7243,128,0.0130,False
7,maxlen,40,0.7243,40,0.0384,True
8,min_freq,3,0.7243,3,0.0207,False
9,dropout,0.0,0.7243,0.0,0.0097,False


**Observación:**

- Solo dos decisiones superan el rango de error: la tasa de aprendizaje (rango de 2,46 pp) y `maxlen` (3,84 pp). En ambas se mantuvo el valor base. El resto quedó en empate técnico.
- Respecto del base cambiaron la celda (LSTM), las capas (3) y los pesos de clase. La mayor parte de la mejora viene de la celda: la LSTM con el resto de la configuración base obtiene 0,7154, 2,98 pp sobre la media del base (0,6856). Las 3 capas suman 0,89 pp y los pesos restan 0,65 pp en esta semilla.
- La mejora de +3,22 pp corresponde a una sola ejecución del modelo final; la comparación con 5 semillas está en la sección 17.4.

## 16. Síntesis de los experimentos

Cada punto es una variante; el eje horizontal es su distancia en F1 macro a la mejor variante de su experimento. Las variantes a la izquierda de la línea amarilla están dentro del rango de error (**empate técnico** con la mejor); el símbolo marca la variante elegida. La tabla completa con todas las columnas queda en el respaldo (`experimentos_todas_las_ejecuciones.csv`).

In [86]:
# ============================================================
# Síntesis gráfica de todos los experimentos
# ============================================================
resumen_general = pd.concat(resultados_experimentos, ignore_index=True)
resumen_general["eleccion"] = ["elegida" if decisiones.get(e) == v else "otra"
                                for e, v in zip(resumen_general["experimento"], resumen_general["variante"])]
fig = px.scatter(resumen_general, x="diferencia_vs_mejor", y="experimento", color="eleccion", symbol="eleccion",
                 text="variante", color_discrete_map={"elegida": AMARILLO, "otra": COLORES[0]},
                 hover_data=["f1_macro_val", "loss_val", "parametros", "segundos_por_epoca", "origen"],
                 title="Distancia de cada variante a la mejor de su experimento (F1 macro de validación)",
                 labels={"diferencia_vs_mejor": "diferencia con la mejor variante", "eleccion": "variante"})
fig.add_vline(x=MARGEN_ERROR, line_dash="dash", line_color=AMARILLO, annotation_text="rango de error")
fig.update_traces(textposition="top center")
fig.update_layout(height=60 * resumen_general["experimento"].nunique() + 150, yaxis={"categoryorder": "array",
                  "categoryarray": list(dict.fromkeys(resumen_general["experimento"]))[::-1]})
fig.show()

**Observación:** 9 de los 11 experimentos terminaron en empate técnico; solo la tasa de aprendizaje y `maxlen` tienen variantes a la derecha del rango de error. En 10 experimentos la variante elegida fue la de mayor F1; en los pesos de clase se eligió la segunda, por contexto (sección 12.2).

## 17. Verificaciones de la dependencia del orden

La selección de un hiperparámetro a la vez es *greedy*: cada decisión se tomó con los valores elegidos hasta ese momento y no explora interacciones. Los experimentos anteriores no se rehacen; aquí se prueban, sin usar test, las combinaciones donde la interacción es más probable, y el modelo final con las mismas 5 semillas que el base:

| Verificación | Sospecha |
|---|---|
| 17.1 GRU con la configuración final | La celda se eligió al inicio con una diferencia dentro del margen; las decisiones posteriores se ajustaron solo para la LSTM |
| 17.2 Tasa 0,003 con lote 256 | Tasa y lote interactúan: un lote más grande da menos actualizaciones por época y tolera una tasa mayor |
| 17.3 Units 128 con Dropout 0,2 | La capacidad se eligió sin regularización; con Dropout, una red más grande podría generalizar mejor |
| 17.4 Configuración final con 5 semillas | Comparación justa con la media de 5 semillas del base |
| 17.5 Cadena de decisiones con 2 capas | Al ampliar el rango de capas cambió la configuración con que se tomaron las decisiones siguientes |

In [87]:
# ============================================================
# Verificaciones 17.1 a 17.3
# ============================================================
VERIFICACIONES = {
    "17.1 GRU con la configuración final": dict(celda="GRU"),
    "17.2 Tasa 0,003 con lote 256": dict(learning_rate=0.003, batch_size=256),
    "17.3 Units 128 con Dropout 0,2": dict(units=128, dropout=0.2),
}
filas_verificacion = [{"verificacion": "Configuración final", **fila_final}]
for nombre, cambios in VERIFICACIONES.items():
    fila, _, _ = entrenar_y_registrar({**CONFIG_FINAL, **cambios})
    print(f"{nombre}: {fila['origen']}")
    filas_verificacion.append({"verificacion": nombre, **fila})
tabla_verificacion = pd.DataFrame(filas_verificacion)
tabla_verificacion["diferencia_vs_final"] = tabla_verificacion["f1_macro_val"] - fila_final["f1_macro_val"]
tabla_verificacion["supera_margen"] = tabla_verificacion["diferencia_vs_final"].abs() > MARGEN_ERROR

17.1 GRU con la configuración final: guardado
17.2 Tasa 0,003 con lote 256: guardado
17.3 Units 128 con Dropout 0,2: guardado


In [88]:
# ============================================================
# Resultados de las verificaciones 17.1 a 17.3
# ============================================================
display(tabla_verificacion[["verificacion", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val",
                            "mejor_epoca", "parametros", "segundos_por_epoca", "diferencia_vs_final",
                            "supera_margen", "origen"]].round(4))

,verificacion,f1_macro_val,exactitud_val,loss_val,brecha_train_val,mejor_epoca,parametros,segundos_por_epoca,diferencia_vs_final,supera_margen,origen
0,Configuración final,0.7178,0.7508,0.6214,0.0353,4,1377028,8.9659,0.0000,False,entrenado
1,17.1 GRU con la configuración final,0.7171,0.7488,0.6247,0.0392,4,1347972,8.4975,-0.0007,False,guardado
2,"17.2 Tasa 0,003 con lote 256",0.7145,0.7443,0.6388,0.0438,4,1377028,2.7779,-0.0033,False,guardado
3,"17.3 Units 128 con Dropout 0,2",0.7248,0.7539,0.6185,0.0328,4,1657348,9.5502,0.0070,False,guardado


**Observación:**

- Las tres verificaciones quedan en empate con la configuración final: la GRU queda 0,07 pp abajo, la tasa 0,003 con lote 256 queda 0,33 pp abajo y 128 unidades con Dropout 0,2 queda 0,70 pp arriba. La selección secuencial no dejó fuera una combinación claramente mejor.
- La combinación de tasa 0,003 y lote 256 es 3,2 veces más rápida (2,8 s contra 8,9 s por época) con el mismo F1: es una alternativa eficiente.

**Hallazgo:** con 3 capas y pesos de clase, la LSTM y la GRU son equivalentes (0,07 pp de diferencia). La ventaja de las celdas con compuertas sobre la RNN simple sí se sostiene (sección 9.2), pero la elección entre LSTM y GRU no es concluyente en la configuración final.

### 17.4 Configuración final con 5 semillas

Se reentrena `CONFIG_FINAL` con las mismas semillas adicionales que el base (sección 8) para comparar medias en igualdad de condiciones.

In [89]:
# ============================================================
# Configuración final con las mismas semillas que el base
# ============================================================
filas_final_semillas = [{"semilla": SEED, **fila_final}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_FINAL, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_final_semillas.append({"semilla": semilla, **fila})
tabla_final_semillas = pd.DataFrame(filas_final_semillas)

Semilla 1: guardado
Semilla 2: guardado
Semilla 3: guardado
Semilla 4: guardado


In [90]:
# ============================================================
# Comparación justa: base vs. final (media de 5 semillas)
# ============================================================
comparacion_justa = pd.DataFrame({
    "base": tabla_variabilidad[["f1_macro_val", "exactitud_val", "loss_val"]].agg(["mean", "std", "min", "max"]).stack(),
    "final": tabla_final_semillas[["f1_macro_val", "exactitud_val", "loss_val"]].agg(["mean", "std", "min", "max"]).stack(),
}).swaplevel().sort_index()
comparacion_justa["diferencia"] = comparacion_justa["final"] - comparacion_justa["base"]
display(comparacion_justa.round(4))

base   final  diferencia
exactitud_val max   0.7661  0.7617     -0.0044
              mean  0.7611  0.7518     -0.0094
              min   0.7529  0.7459     -0.0070
              std   0.0050  0.0065      0.0014
f1_macro_val  max   0.6975  0.7285      0.0310
              mean  0.6856  0.7203      0.0347
              min   0.6740  0.7169      0.0429
              std   0.0088  0.0049     -0.0039
loss_val      max   0.6343  0.6589      0.0246
              mean  0.6296  0.6321      0.0025
              min   0.6247  0.6119     -0.0128
              std   0.0043  0.0177      0.0134

**Hallazgo:**

- Con 5 semillas, la configuración final obtiene un F1 macro medio de **0,7203** (desviación de 0,49 pp), contra 0,6856 del base: **+3,47 pp**, por encima del rango de error (2,35 pp). La mejora es real.
- Esta vez la semilla 42 (0,7178) no es la mejor de las cinco (máximo de 0,7285): la estimación de una sola ejecución (+3,22 pp) quedó incluso por debajo de la media.
- La exactitud media baja 0,94 pp y la pérdida de validación no mejora (+0,0025): los pesos priorizan las clases pequeñas a costa de la mayoritaria, y la pérdida de validación se calcula sin pesos.
- La configuración final es más estable que el base en F1 (desviación de 0,49 pp contra 0,88 pp).
- En la ejecución previa de este notebook sin pesos de clase (respaldo de esa ejecución), la misma configuración obtuvo una media de 0,7116 con 5 semillas. Los pesos balanceados elevan el F1 medio en 0,87 pp y lo hacen más estable, lo que respalda la decisión de la sección 12.2.

### 17.5 Efecto del orden: la cadena de decisiones con 2 capas

Las corridas de este notebook se ejecutaron primero con un rango de capas de 1 y 2. Ganó 2 capas, que estaba en el **borde** del rango, y con esa configuración se tomaron las decisiones de los experimentos 11.3 a 12.2 (lectura bidireccional, embedding, largo máximo, vocabulario, Dropout y pesos de clase).

Después se incorporó la regla de ampliar el rango cuando el ganador queda en el borde (función `ampliar_rango`, sección 5.3). Al probar 3 y 4 capas, ganó 3, y los experimentos 11.3 a 12.2 se repitieron con esa configuración: son los que aparecen en las secciones 11 y 12.

Las corridas de la primera cadena quedaron guardadas (sección 2.1), así que aquí se reconstruye **sin entrenar** qué se eligió en cada experimento con 2 capas, aplicando en todos el criterio automático (mayor F1 macro), y se compara con lo elegido en la cadena actual con 3 capas.

In [91]:
# ============================================================
# Reconstrucción de la cadena con 2 capas desde los resultados guardados
# ============================================================
VARIANTES_POSTERIORES = {
    "bidireccional": variantes_bidireccional, "embed_dim": variantes_embed_dim, "maxlen": variantes_maxlen,
    "min_freq": variantes_min_freq, "dropout": variantes_dropout, "pesos_clase": variantes_pesos_clase,
}
config_cadena = {**dict(historial_experimentos)["capas"], "capas": 2}
tablas_actuales = {t.loc[0, "experimento"]: t for t in resultados_experimentos}
filas_orden = []
for nombre, variantes in VARIANTES_POSTERIORES.items():
    f1_guardados = {}
    for etiqueta, cambios in variantes.items():
        clave = clave_corrida({**config_cadena, **cambios}, SEED, EPOCAS_EXPERIMENTO, PACIENCIA_EXPERIMENTO)
        if clave in RESULTADOS_GUARDADOS:
            f1_guardados[etiqueta] = RESULTADOS_GUARDADOS[clave]["f1_macro_val"]
    if not f1_guardados:
        print(f"[{nombre}] sin corridas guardadas para la cadena con 2 capas")
        continue
    elegida_2 = max(f1_guardados, key=f1_guardados.get)
    config_cadena.update(variantes[elegida_2])
    tabla = tablas_actuales[nombre]
    elegida_3 = decisiones[nombre]
    filas_orden.append({
        "experimento": nombre,
        "elegida_con_2_capas": elegida_2, "f1_con_2_capas": f1_guardados[elegida_2],
        "elegida_con_3_capas": elegida_3,
        "f1_con_3_capas": tabla.loc[tabla["variante"] == elegida_3, "f1_macro_val"].iloc[0],
        "misma_decision": elegida_2 == elegida_3,
    })
tabla_orden = pd.DataFrame(filas_orden)

In [92]:
# ============================================================
# Decisiones de ambas cadenas
# ============================================================
display(tabla_orden.round(4))

,experimento,elegida_con_2_capas,f1_con_2_capas,elegida_con_3_capas,f1_con_3_capas,misma_decision
0,bidireccional,unidireccional,0.7179,unidireccional,0.7243,True
1,embed_dim,64,0.7205,128,0.7243,False
2,maxlen,40,0.7205,40,0.7243,True
3,min_freq,1,0.7214,3,0.7243,False
4,dropout,0.0,0.7214,0.0,0.7243,True
5,pesos_clase,sin pesos,0.7214,balanceados,0.7178,False


**Hallazgo:**

- De las seis decisiones posteriores a la de capas, tres coinciden en ambas cadenas (lectura unidireccional, `maxlen = 40` y sin Dropout) y tres cambian: `embed_dim` (64 con 2 capas, 128 con 3), `min_freq` (1 con 2 capas, 3 con 3) y los pesos de clase (con 2 capas se aplicó el criterio automático, sin pesos; con 3 se eligieron los balanceados por contexto).
- La cadena con 2 capas termina con un F1 macro de 0,7214 y la de 3 capas, antes de los pesos de clase, con 0,7243: una diferencia de 0,29 pp, dentro del rango de error.
- El orden cambió **qué valores se eligen** entre opciones empatadas, pero no el desempeño de forma significativa. Como casi todas las decisiones de la selección fueron empates técnicos, las dos cadenas son configuraciones equivalentes.

## 18. Conclusiones generales

- **Elección de la celda:** las celdas con compuertas superan a la RNN simple donde la teoría lo predice, en turnos largos (+5,7 pp la LSTM y +7,4 pp la GRU en turnos de más de 20 tokens). En el total, las diferencias quedan dentro del rango de error, y en la configuración final la LSTM y la GRU son equivalentes.
- **Tasa de aprendizaje, tamaño del lote y épocas:** la tasa es la única de las tres que supera el rango de error (0,003 cae 2,46 pp; 0,0001 necesita 14 épocas para converger). El lote cambia sobre todo el costo (de 2,1 a 12,4 s por época) y no el F1. Sin Early Stopping, la pérdida de validación sube de 0,59 a 1,31 entre las épocas 5 y 30, así que un tope de 25 épocas con paciencia 3 basta.
- **Arquitectura y regularización:** tras ampliar el rango, 3 capas resultó la mejor y quedó en el interior. `maxlen = 40` es la otra decisión que supera el rango de error (20 tokens cuesta 3,84 pp). El Dropout no mejora el F1. Los pesos balanceados se eligieron por contexto: suben la exhaustividad de *commissive* de 0,40 a 0,63.
- **Dependencia del orden:** ampliar el rango de capas de 2 a 3 cambió las elecciones de `embed_dim` (de 64 a 128) y `min_freq` (de 1 a 3) en los experimentos siguientes (sección 17.5), pero el F1 de ambas cadenas difiere solo 0,29 pp, dentro del rango de error. Las verificaciones de la sección 17 también quedan en empate con la configuración final: el orden cambia qué valores se eligen entre opciones empatadas, pero no deja fuera una combinación claramente mejor.
- **Desempeño:** en test, exactitud 0,773 y F1 macro 0,716, frente a 0,728 y 0,423 de la regla «?». Con 5 semillas, la mejora real sobre el base es de 3,47 pp de F1 macro. *commissive* sigue siendo la clase más débil, ahora por sobrepredicción (precisión de 0,38).
- **Limitaciones:**
  - Cada turno se clasifica aislado, sin el contexto del diálogo, que explica buena parte de los errores.
  - El 26 % de los turnos de validación y test aparece textualmente en `train`, y 69 diálogos de validación y 94 de test son idénticos a uno de `train`; esto puede inflar levemente las métricas.
  - Se evaluaron muchas variantes sobre la misma validación; las 5 semillas del modelo final atenúan, pero no eliminan, ese sesgo optimista.

### 18.1 Mejoras propuestas

- **Agregar el turno anterior como contexto.** El acto depende de la conversación: una respuesta a una pregunta suele ser *inform* o *commissive*, y hoy el modelo ve cada turno aislado.
- **Aprovechar la emoción de cada turno (aprendizaje multitarea).** Cada turno del dataset trae dos etiquetas: su acto y su emoción (`emotion`, 7 clases). Hoy solo se usa el acto. La mejora consiste en entrenar **un mismo encoder recurrente con dos salidas**: una capa que predice el acto (4 clases) y otra que predice la emoción (7 clases), sumando ambas pérdidas. La tarea adicional obliga al encoder a aprender rasgos más ricos del turno (tono, cortesía), que también pueden ayudar a distinguir actos. Como el 82,8 % de los turnos es «no emotion», la salida de emoción necesita pesos de clase y evaluarse con F1 macro.
- **Usar embeddings preentrenados** (por ejemplo, GloVe) en vez de aprenderlos desde cero, para mejorar las palabras poco frecuentes.
- **Reemplazar la capa recurrente por un encoder Transformer** como clasificador y comparar con la LSTM.

## 19. Conclusión final

El modelo final, una LSTM de 3 capas con pesos de clase balanceados, clasifica el acto de diálogo con un F1 macro de 0,716 en test y supera ampliamente a una regla basada solo en la puntuación (0,423). El análisis por largo del turno justifica una celda con compuertas frente a la RNN simple, y las decisiones se tomaron con un rango de error medido, verificaciones de la dependencia del orden y una comparación con 5 semillas que confirma una mejora real de 3,47 pp sobre el modelo base. Su principal limitación es clasificar cada turno sin el contexto del diálogo, que es la mejora con más potencial.

## 20. Referencias

- Li, Y., Su, H., Shen, X., Li, W., Cao, Z. y Niu, S. (2017). *DailyDialog: A Manually Labelled Multi-turn Dialogue Dataset*. IJCNLP.
- Hochreiter, S. y Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation.
- Cho, K. et al. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. EMNLP (GRU).
- Kingma, D. P. y Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR.
- Documentación de PyTorch: `torch.nn.RNN`, `torch.nn.LSTM`, `torch.nn.GRU`, `pack_padded_sequence`.
- *Del embedding a las RNN* y *Repaso RNN y LSTM*. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).
- *RNN simple: efecto de los hiperparámetros* (`RNN_hiperparametros_DLY0100`). Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).
- *Implementación de un Transformer con PyTorch* (`GP_3_5_4`), del que provienen las clases `Vocab` y `BucketSampler`. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).

## 21. Anexo A — Síntesis visual para la presentación

Gráficos de los resultados principales, con formato 16:9 para la presentación. Se generan desde las variables ya calculadas, sin entrenar nada. Cada gráfico se guarda en `respaldo_ep3_rnn/graficos/` como **HTML interactivo**; el botón de cámara del gráfico descarga un **PNG de 1600 × 900**. Si la librería `kaleido` está disponible, el PNG también se guarda directamente.

In [93]:
# ============================================================
# Estilo y exportación de los gráficos de la presentación
# ============================================================
RUTA_RESPALDO = "respaldo_ep3_rnn"
RUTA_GRAFICOS = f"{RUTA_RESPALDO}/graficos"
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
AZUL_MARINO = "#0E2747"
ANCHO_PNG, ALTO_PNG = 1600, 900


def estilo_presentacion(fig, titulo, subtitulo=None):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:15px;color:#555555'>{subtitulo}</span>"
    fig.update_layout(template="plotly_white", title={"text": texto, "x": 0.02},
                      font={"family": "Calibri, Arial", "size": 16, "color": AZUL_MARINO},
                      height=560, margin={"t": 110, "l": 80, "r": 40, "b": 80},
                      legend={"orientation": "h", "y": -0.18, "title": None})
    return fig


def exportar(fig, nombre):
    fig.write_html(f"{RUTA_GRAFICOS}/{nombre}.html", include_plotlyjs="cdn",
                   config={"toImageButtonOptions": {"format": "png", "width": ANCHO_PNG, "height": ALTO_PNG,
                                                    "scale": 2, "filename": nombre}})
    try:
        fig.write_image(f"{RUTA_GRAFICOS}/{nombre}.png", width=ANCHO_PNG, height=ALTO_PNG, scale=2)
        print(f"{nombre}: guardado en HTML y PNG")
    except Exception:
        print(f"{nombre}: guardado en HTML (PNG desde el botón de cámara del gráfico)")

### A.1 Modelo base frente a modelo final (5 semillas)

Cada punto es una semilla; la línea marca la media y la franja amarilla, el rango de error del modelo base. Mensaje: la mejora es real y supera el ruido.

In [94]:
# ============================================================
# A.1 Base frente a final con 5 semillas
# ============================================================
base = tabla_variabilidad["f1_macro_val"].to_numpy()
final = tabla_final_semillas["f1_macro_val"].to_numpy()
fig = go.Figure()
fig.add_hrect(y0=base.min(), y1=base.max(), fillcolor=AMARILLO, opacity=0.25, line_width=0,
              annotation_text=f"rango de error del base ({MARGEN_ERROR * 100:.2f} pp)", annotation_position="top left")
for etiqueta, valores, color in [("Base: RNN simple", base, COLORES[2]), ("Final: LSTM de 3 capas con pesos", final, AZUL_MARINO)]:
    fig.add_trace(go.Scatter(x=[etiqueta] * len(valores), y=valores, mode="markers", name=etiqueta,
                             marker={"size": 14, "color": color, "line": {"width": 1, "color": "white"}}))
    fig.add_trace(go.Scatter(x=[etiqueta], y=[valores.mean()], mode="markers+text", showlegend=False,
                             marker={"size": 40, "symbol": "line-ew-open", "color": AZUL_MARINO, "line": {"width": 4}},
                             text=[f"media {valores.mean():.4f}"], textposition="middle right"))
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, f"Mejora real de {(final.mean() - base.mean()) * 100:+.2f} pp en F1 macro",
                    "Cinco semillas por configuración, mismas semillas en ambas")
exportar(fig, "01_base_vs_final_semillas")
fig.show()

01_base_vs_final_semillas: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.2 Impacto de cada decisión

Diferencia entre la mejor y la peor variante de cada experimento, frente al rango de error. Mensaje: solo la tasa de aprendizaje y el largo máximo superan el ruido.

In [95]:
# ============================================================
# A.2 Impacto de cada decisión frente al rango de error
# ============================================================
datos = tabla_decisiones.assign(
    rango_pp=tabla_decisiones["rango_entre_variantes"] * 100,
    resultado=np.where(tabla_decisiones["supera_margen"], "supera el rango de error", "empate técnico"))
fig = px.bar(datos, x="experimento", y="rango_pp", color="resultado", text=datos["rango_pp"].round(2),
             color_discrete_map={"supera el rango de error": AZUL_MARINO, "empate técnico": COLORES[2]},
             labels={"rango_pp": "mejor − peor variante (pp de F1 macro)", "experimento": ""})
fig.add_hline(y=MARGEN_ERROR * 100, line_dash="dash", line_color=AMARILLO,
              annotation_text=f"rango de error: {MARGEN_ERROR * 100:.2f} pp")
estilo_presentacion(fig, "Qué decisiones superan el ruido", "Diferencia entre la mejor y la peor variante de cada experimento")
exportar(fig, "02_impacto_decisiones")
fig.show()

02_impacto_decisiones: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.3 Decisión clave: celda según el largo del turno

F1 macro de validación de cada celda por tramo de largo. Mensaje: las celdas con compuertas ganan en los turnos largos, como predice la teoría.

In [96]:
# ============================================================
# A.3 Celda recurrente según el largo del turno
# ============================================================
fig = px.bar(tabla_tramos, x="tramo", y="f1_macro", color="celda", barmode="group", text=tabla_tramos["f1_macro"].round(3),
             color_discrete_map={"RNN simple": COLORES[2], "GRU": COLORES[1], "LSTM": AZUL_MARINO},
             category_orders={"tramo": TRAMOS, "celda": ["RNN simple", "GRU", "LSTM"]},
             labels={"f1_macro": "F1 macro (validación)", "tramo": "tokens por turno"})
fig.update_yaxes(range=[0.5, 0.8])
estilo_presentacion(fig, "Las compuertas importan en los turnos largos",
                    "RNN simple, GRU y LSTM con la configuración base, por largo del turno")
exportar(fig, "03_celda_por_largo")
fig.show()

03_celda_por_largo: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.4 Tasa de aprendizaje

F1 macro y mejor época de cada tasa. Mensaje: una tasa alta converge en 2 épocas y rinde peor; una baja necesita muchas más épocas.

In [97]:
# ============================================================
# A.4 Tasa de aprendizaje: F1 y velocidad de convergencia
# ============================================================
datos = tabla_learning_rate.assign(tasa=tabla_learning_rate["variante"].astype(float)).sort_values("tasa")
colores = [AZUL_MARINO if v == decisiones["learning_rate"] else COLORES[2] for v in datos["variante"]]
fig = make_subplots(rows=1, cols=2, subplot_titles=["F1 macro (validación)", "Mejor época"])
fig.add_trace(go.Bar(x=datos["variante"], y=datos["f1_macro_val"], marker_color=colores,
                     text=datos["f1_macro_val"].round(4), showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=datos["variante"], y=datos["mejor_epoca"], marker_color=colores,
                     text=datos["mejor_epoca"], showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="tasa de aprendizaje", type="category")
fig.update_yaxes(range=[0.65, 0.73], row=1, col=1)
estilo_presentacion(fig, "Tasa de aprendizaje: velocidad frente a calidad", "En azul oscuro, la tasa elegida")
exportar(fig, "04_tasa_aprendizaje")
fig.show()

04_tasa_aprendizaje: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.5 Épocas sin Early Stopping

Pérdida y F1 macro durante 30 épocas sin detener el entrenamiento. Mensaje: desde la época 5 la pérdida de validación sube (sobreajuste); el Early Stopping detiene a tiempo.

In [98]:
# ============================================================
# A.5 Épocas: sobreajuste sin Early Stopping
# ============================================================
epocas = list(range(1, len(historia_epocas["loss_val"]) + 1))
mejor = int(np.argmin(historia_epocas["loss_val"])) + 1
fig = make_subplots(rows=1, cols=2, subplot_titles=["Pérdida", "F1 macro (validación)"])
fig.add_trace(go.Scatter(x=epocas, y=historia_epocas["loss_train"], name="entrenamiento", line_color=COLORES[2]), row=1, col=1)
fig.add_trace(go.Scatter(x=epocas, y=historia_epocas["loss_val"], name="validación", line_color=AZUL_MARINO), row=1, col=1)
fig.add_trace(go.Scatter(x=epocas, y=historia_epocas["f1_val"], name="F1 macro de validación", line_color=COLORES[1]), row=1, col=2)
for col in [1, 2]:
    fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
    fig.update_xaxes(title_text="época", row=1, col=col)
estilo_presentacion(fig, f"Más épocas no ayudan: mínima pérdida de validación en la época {mejor}",
                    "Configuración del experimento 10.3 entrenada 30 épocas sin Early Stopping")
exportar(fig, "05_epocas_sin_early_stopping")
fig.show()

05_epocas_sin_early_stopping: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.6 Tamaño del lote: costo frente a F1

Tiempo por época y F1 macro de cada lote. Mensaje: todos empatan en F1; el lote cambia sobre todo el costo.

In [99]:
# ============================================================
# A.6 Tamaño del lote: costo frente a desempeño
# ============================================================
datos = tabla_batch_size.assign(eleccion=np.where(tabla_batch_size["variante"] == decisiones["batch_size"], "elegido", "otro"))
fig = px.scatter(datos, x="segundos_por_epoca", y="f1_macro_val", text="variante", color="eleccion",
                 color_discrete_map={"elegido": AZUL_MARINO, "otro": COLORES[1]},
                 labels={"segundos_por_epoca": "segundos por época", "f1_macro_val": "F1 macro (validación)"})
fig.add_hrect(y0=datos["f1_macro_val"].max() - MARGEN_ERROR, y1=datos["f1_macro_val"].max(), fillcolor=AMARILLO,
              opacity=0.2, line_width=0, annotation_text="empate técnico con el mejor", annotation_position="bottom left")
fig.update_traces(marker={"size": 18}, textposition="top center")
estilo_presentacion(fig, "Tamaño del lote: mismo F1, distinto costo", "Etiqueta: tamaño del lote")
exportar(fig, "06_lote_costo_vs_f1")
fig.show()

06_lote_costo_vs_f1: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.7 Pesos de clase: precisión y exhaustividad por clase

Métricas por clase en validación, sin pesos y con pesos balanceados. Mensaje: los pesos suben la exhaustividad de *commissive* a costa de su precisión.

In [100]:
# ============================================================
# A.7 Pesos de clase: métricas por clase
# ============================================================
datos = (pd.DataFrame(filas_por_clase)
         .rename(columns={"precision": "precisión", "exhaustividad": "exhaustividad", "f1": "F1"})
         .melt(id_vars=["variante", "acto"], var_name="metrica", value_name="valor"))
fig = px.bar(datos, x="acto", y="valor", color="metrica", barmode="group", facet_col="variante",
             category_orders={"acto": ACTOS, "variante": ["sin pesos", "balanceados"], "metrica": ["precisión", "exhaustividad", "F1"]},
             color_discrete_sequence=[COLORES[2], COLORES[1], AZUL_MARINO], text=datos["valor"].round(2),
             labels={"valor": "", "acto": ""})
fig.update_yaxes(range=[0, 1])
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("variante=", "")))
estilo_presentacion(fig, "Pesos de clase: más exhaustividad en commissive, menos precisión",
                    "Validación; precisión, exhaustividad y F1 por acto de diálogo")
exportar(fig, "07_pesos_por_clase")
fig.show()

07_pesos_por_clase: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.8 Resultado en test frente a las líneas base

Exactitud, F1 macro y F1 ponderado en test. Mensaje: el modelo supera ampliamente a la regla basada en la puntuación.

In [101]:
# ============================================================
# A.8 Resultado en test frente a las líneas base
# ============================================================
datos = (tabla_test[["modelo", "exactitud", "f1_macro", "f1_ponderado"]]
         .rename(columns={"f1_macro": "F1 macro", "f1_ponderado": "F1 ponderado"})
         .melt(id_vars="modelo", var_name="metrica", value_name="valor"))
fig = px.bar(datos, x="metrica", y="valor", color="modelo", barmode="group", text=datos["valor"].round(3),
             color_discrete_sequence=[COLORES[2], COLORES[1], AZUL_MARINO], labels={"valor": "", "metrica": ""})
fig.update_yaxes(range=[0, 1])
estilo_presentacion(fig, "Resultado en test", "Modelo final frente a la clase mayoritaria y a la regla del signo «?»")
exportar(fig, "08_test_vs_lineas_base")
fig.show()

08_test_vs_lineas_base: guardado en HTML (PNG desde el botón de cámara del gráfico)


### A.9 Matriz de confusión en test

Proporción por fila y cantidad de turnos. Mensaje: el modelo distingue bien las preguntas y confunde los actos sin marca formal.

In [102]:
# ============================================================
# A.9 Matriz de confusión en test
# ============================================================
conteos = confusion_matrix(y_test, y_pred_test)
texto = [[f"{matriz[i, j]:.2f}<br>({conteos[i, j]})" for j in range(len(ACTOS))] for i in range(len(ACTOS))]
fig = px.imshow(matriz, x=ACTOS, y=ACTOS, color_continuous_scale=[[0, "#FFFFFF"], [1, AZUL_MARINO]], zmin=0, zmax=1,
                labels={"x": "acto predicho", "y": "acto real", "color": "proporción"})
fig.update_traces(text=texto, texttemplate="%{text}")
estilo_presentacion(fig, "Dónde se equivoca el modelo", "Test; proporción por fila y cantidad de turnos entre paréntesis")
exportar(fig, "09_matriz_confusion")
fig.show()

09_matriz_confusion: guardado en HTML (PNG desde el botón de cámara del gráfico)


## 22. Anexo B — Herramientas y tecnologías de apoyo utilizadas

En el desarrollo de este trabajo se utilizaron las siguientes herramientas y tecnologías:

**Entornos de ejecución:**

- Google Colab (GPU NVIDIA T4) — entorno de ejecución del notebook.
- Visual Studio Code con la extensión Google Colab — edición y ejecución del notebook conectado a Colab.

**Asistente de IA:**

- Claude (Opus 5.5), Anthropic — apoyo en la estructuración del notebook, el código y la redacción.

**Herramienta de estudio:**

- Transformer Explainer — [https://poloclub.github.io/transformer-explainer/](https://poloclub.github.io/transformer-explainer/)

**Repositorio:**

- GitHub — [https://github.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY0100_003D](https://github.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY0100_003D)

**Librerías y frameworks (sitios oficiales):**

- PyTorch — [https://pytorch.org/](https://pytorch.org/)
- scikit-learn — [https://scikit-learn.org/](https://scikit-learn.org/)
- NumPy — [https://numpy.org/](https://numpy.org/)
- Pandas — [https://pandas.pydata.org/](https://pandas.pydata.org/)
- Plotly (Python) — [https://plotly.com/python/](https://plotly.com/python/)

## 23. Respaldo de resultados

Se guardan las tablas, la historia, el modelo final y los gráficos del anexo A en `respaldo_ep3_rnn/` y se comprime en `respaldo_ep3_rnn.zip`. Se incluye `resultados_rnn_guardados.json` actualizado con todas las corridas de esta ejecución, para reutilizarlas en una próxima. En Colab web, el `.zip` se descarga con `from google.colab import files; files.download("respaldo_ep3_rnn.zip")`.

In [103]:
# ============================================================
# Respaldo de resultados
# ============================================================
RUTA_RESPALDO = "respaldo_ep3_rnn"
os.makedirs(RUTA_RESPALDO, exist_ok=True)

resumen_general.to_csv(f"{RUTA_RESPALDO}/experimentos_todas_las_ejecuciones.csv", index=False)
tabla_variabilidad.to_csv(f"{RUTA_RESPALDO}/variabilidad.csv", index=False)
estadisticos_variabilidad.to_csv(f"{RUTA_RESPALDO}/estadisticos_variabilidad.csv")
tabla_tramos.to_csv(f"{RUTA_RESPALDO}/f1_por_largo.csv", index=False)
tabla_epocas.to_csv(f"{RUTA_RESPALDO}/epocas_sin_early_stopping.csv")
config_base_vs_final.to_csv(f"{RUTA_RESPALDO}/config_base_vs_final.csv")
tabla_decisiones.to_csv(f"{RUTA_RESPALDO}/decisiones.csv", index=False)
tabla_verificacion.to_csv(f"{RUTA_RESPALDO}/verificaciones_orden.csv", index=False)
tabla_final_semillas.to_csv(f"{RUTA_RESPALDO}/final_5_semillas.csv", index=False)
tabla_orden.to_csv(f"{RUTA_RESPALDO}/cadena_2_capas_vs_3_capas.csv", index=False)
comparacion_justa.to_csv(f"{RUTA_RESPALDO}/comparacion_base_final.csv")
tabla_lineas_base_val.to_csv(f"{RUTA_RESPALDO}/lineas_base_validacion.csv", index=False)
tabla_test.to_csv(f"{RUTA_RESPALDO}/metricas_test.csv", index=False)
reporte.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase_test.csv")
reporte_val.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase_validacion.csv")
pd.DataFrame(matriz, index=ACTOS, columns=ACTOS).to_csv(f"{RUTA_RESPALDO}/matriz_confusion.csv")
with open(f"{RUTA_RESPALDO}/historia_final.json", "w") as f:
    json.dump(historia_final, f)
np.save(f"{RUTA_RESPALDO}/y_test.npy", y_test)
np.save(f"{RUTA_RESPALDO}/y_test_proba.npy", pred_test["probas"])
torch.save({"config": CONFIG_FINAL, "estado": modelo_final.state_dict(), "itos": vocab_final.get_itos()},
           f"{RUTA_RESPALDO}/mejor_modelo_rnn.pt")

with open(f"{RUTA_RESPALDO}/resultados_rnn_guardados.json", "w", encoding="utf-8") as f:
    json.dump({**RESULTADOS_GUARDADOS, **CORRIDAS_NUEVAS}, f, ensure_ascii=False, indent=1,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

shutil.make_archive(RUTA_RESPALDO, "zip", RUTA_RESPALDO)
print("Respaldo guardado en", RUTA_RESPALDO + ".zip")

Respaldo guardado en respaldo_ep3_rnn.zip


## 24. Descripción de las salidas

La celda de respaldo (sección 23) genera `respaldo_ep3_rnn.zip`. En la carpeta de entrega, el `.zip` va **sin los pesos** del modelo y el archivo `.pt` se deja junto a él.

| Archivo | Contenido | Sección |
|---|---|---|
| `lineas_base_validacion.csv` | Clase mayoritaria y regla «?» en validación | 3 |
| `variabilidad.csv` y `estadisticos_variabilidad.csv` | Corridas del modelo base con 5 semillas y sus estadísticos (rango de error) | 8 |
| `f1_por_largo.csv` | F1 macro de cada celda según el largo del turno | 9 |
| `epocas_sin_early_stopping.csv` | Métricas por época sin Early Stopping | 10 |
| `experimentos_todas_las_ejecuciones.csv` | Todas las variantes de los experimentos con todas sus métricas | 9 a 12 |
| `historia_final.json` | Pérdida y métricas por época del modelo final | 13 |
| `metricas_test.csv` | Métricas del modelo final y de las líneas base en test | 14 |
| `metricas_por_clase_test.csv` y `metricas_por_clase_validacion.csv` | Precisión, exhaustividad y F1 por acto | 14 |
| `matriz_confusion.csv` | Matriz de confusión de test | 14 |
| `y_test.npy` y `y_test_proba.npy` | Etiquetas reales de test y probabilidades predichas para cada acto | 14 |
| `config_base_vs_final.csv` y `decisiones.csv` | Hiperparámetros del base y del final, y aporte de cada decisión | 15 |
| `verificaciones_orden.csv` y `final_5_semillas.csv` | Verificaciones del orden y configuración final con 5 semillas | 17 |
| `comparacion_base_final.csv` | Comparación base frente a final con 5 semillas | 17 |
| `cadena_2_capas_vs_3_capas.csv` | Decisiones de la cadena con 2 capas frente a la de 3 | 17.5 |
| `resultados_rnn_guardados.json` | Métricas de todas las corridas, por configuración, semilla, épocas y paciencia | 2 |
| `graficos/*.html` | Los 9 gráficos del anexo A (PNG desde el botón de cámara de cada gráfico) | 21 |
| `mejor_modelo_rnn.pt` | Pesos del modelo final, con su configuración y su vocabulario (`itos`) | 13 |

**Reutilizar las corridas:** con `resultados_rnn_guardados.json` en la carpeta de trabajo de Colab (o en la raíz del repositorio), el notebook solo entrena los modelos que necesita completos y las configuraciones nuevas.

**Cargar el modelo sin entrenar:** ejecutar las secciones 2 a 6 (ninguna entrena) y luego:

```python
ckpt = torch.load("mejor_modelo_rnn.pt", map_location=device)
stoi = {t: i for i, t in enumerate(ckpt["itos"])}
modelo = construir_modelo(ckpt["config"], len(stoi), stoi["<pad>"])
modelo.load_state_dict(ckpt["estado"])
modelo.eval()
ids = torch.tensor([[stoi.get(t, stoi["<unk>"]) for t in "can you help me ?".split()]], device=device)
ACTOS[modelo(ids, torch.tensor([ids.shape[1]])).argmax(1).item()]
```

El turno debe tener el mismo formato que los datos: en minúsculas y con la puntuación separada por espacios.